In [1]:
import os
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0 --tf_xla_enable_xla_devices=false"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
tf.config.optimizer.set_jit(False)

print("✅ XLA OFF")
print("GPUs:", tf.config.list_physical_devices("GPU"))

2026-02-21 09:11:10.086225: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771665070.254620      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771665070.305822      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771665070.709465      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771665070.709512      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771665070.709515      23 computation_placer.cc:177] computation placer alr

✅ XLA OFF
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
import pandas as pd
from tensorflow.keras import layers

train_csv = "/kaggle/input/datasets/geleefinal/gelee-nakba/AR-MS.train/annotations.csv"

df = pd.read_csv(train_csv)
df = df[df["text"].notna()].copy()
df["text"] = df["text"].astype(str)
df = df[df["text"].str.len() <= 200].copy()

print("✅ df:", df.shape)
print(df.head(2))

max_label_len = 200

all_texts = df["text"].tolist()
all_chars = sorted(list(set("".join(all_texts))))

char_to_num = layers.StringLookup(vocabulary=all_chars, mask_token=None)
num_to_char = layers.StringLookup(vocabulary=char_to_num.get_vocabulary(), mask_token=None, invert=True)

print("✅ vocab size:", len(all_chars))
print("✅ lookup size:", len(char_to_num.get_vocabulary()))

✅ df: (15962, 2)
                                          image  \
0  732294a1c8ef4d55ad023c877c098e26-0011-67.jpg   
1  cc158b0357334687b8f91d95f4e802c0-0021-62.jpg   

                                                text  
0  أدلى الرئيس عبد الناصر بحديث إلى المستر روبرت ...  
1  في كل سنة ذكرى لهذه الحادثة التي لا تمت إلى ال...  
✅ vocab size: 137
✅ lookup size: 138


I0000 00:00:1771665094.095673      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [3]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

train_images_dir = "/kaggle/input/datasets/geleefinal/gelee-nakba/AR-MS.train/images"

img_width  = 1536
img_height = 64
batch_size = 64

def preprocess_image(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=1, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, [img_height, img_width])

    # RTL
    img = tf.image.flip_left_right(img)
    # time axis
    img = tf.transpose(img, perm=[1, 0, 2])
    return img

def process_example(image_name, text):
    path = tf.strings.join([train_images_dir, "/", image_name])
    img = preprocess_image(path)

    label = char_to_num(tf.strings.unicode_split(text, "UTF-8"))
    label_len = tf.cast(tf.shape(label)[0], tf.int64)

    label = tf.pad(label, [[0, max_label_len - tf.shape(label)[0]]], constant_values=-1)
    label = tf.cast(label, tf.int32)

    # pooling مرتين => 1536/4 = 384
    input_len = tf.cast(img_width // 4, tf.int64)

    return {
        "image": img,
        "label": label,
        "input_length": tf.expand_dims(input_len, 0),
        "label_length": tf.expand_dims(label_len, 0),
    }

x_train, x_val, y_train, y_val = train_test_split(
    df["image"].astype(str).tolist(),
    df["text"].astype(str).tolist(),
    test_size=0.1,
    random_state=42
)

def make_ds(images, labels, training=True):
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    ds = ds.map(process_example, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(2048)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_ds(x_train, y_train, True)
val_ds   = make_ds(x_val, y_val, False)

print("✅ train/val:", len(x_train), len(x_val))

✅ train/val: 14365 1597


In [4]:
from tensorflow.keras import layers, models
import tensorflow as tf

class CTCLayer(layers.Layer):
    def call(self, y_true, y_pred, input_length, label_length):
        loss = tf.keras.backend.ctc_batch_cost(y_true, y_pred, input_length, label_length)
        self.add_loss(loss)
        return y_pred

def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    return x

def build_model(vocab_size):
    inp = layers.Input(shape=(img_width, img_height, 1), name="image", dtype="float32")

    x = conv_block(inp, 64)
    x = layers.MaxPooling2D((2,2))(x)

    x = conv_block(x, 128)
    x = layers.MaxPooling2D((2,2))(x)

    x = conv_block(x, 256)
    x = layers.Dropout(0.15)(x)

    new_shape = (img_width // 4, (img_height // 4) * 256)
    x = layers.Reshape(new_shape)(x)

    x = layers.Dense(384, activation="relu")(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Bidirectional(layers.LSTM(256, return_sequences=True, dropout=0.25))(x)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.25))(x)

    out = layers.Dense(vocab_size + 1, activation="softmax")(x)

    prediction_model = models.Model(inp, out, name="predict_crnn")

    labels = layers.Input(name="label", shape=(max_label_len,), dtype="int32")
    in_len = layers.Input(name="input_length", shape=(1,), dtype="int64")
    lb_len = layers.Input(name="label_length", shape=(1,), dtype="int64")

    out_ctc = CTCLayer(name="ctc_loss")(labels, out, in_len, lb_len)
    train_model = models.Model([inp, labels, in_len, lb_len], out_ctc, name="train_crnn_ctc")
    return prediction_model, train_model

prediction_model, train_model = build_model(vocab_size=len(char_to_num.get_vocabulary()))
print("✅ models ready:", prediction_model.output_shape)

✅ models ready: (None, 384, 139)


In [5]:
import numpy as np, re, os
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping, Callback

ckpt_path = "/kaggle/working/best_final_raw.weights.h5"

vocab = tf.constant(char_to_num.get_vocabulary())
vocab_size = int(vocab.shape[0])

def greedy_decode(pred):
    ilen = np.ones(pred.shape[0]) * pred.shape[1]
    dec, _ = tf.keras.backend.ctc_decode(pred, input_length=ilen, greedy=True)
    seq = dec[0][0]
    seq = seq[seq != -1]
    seq = seq[seq < vocab_size]
    if seq.shape[0] == 0: return ""
    return tf.strings.reduce_join(tf.gather(vocab, seq)).numpy().decode("utf-8")

def light_post(s):
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"\s+([،\.!\?؛:])", r"\1", s)
    return s

sample_imgs = x_val[:3]
sample_txts = y_val[:3]

class SamplePreds(Callback):
    def __init__(self, every=5):
        super().__init__()
        self.every = every

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.every != 0 and (epoch + 1) != 1:
            return
        print("\n🔎 Sample Predictions:")
        for name, gt in zip(sample_imgs, sample_txts):
            path = os.path.join(train_images_dir, name)
            img = preprocess_image(path)
            pred = prediction_model(tf.expand_dims(img,0), training=False).numpy()
            txt = light_post(greedy_decode(pred))
            print(f"🖼️ {name}\nGT  : {gt[:120]}\nPRED: {txt[:120]}\n")

reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1)
early     = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1)
ckpt      = ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True, save_weights_only=True, verbose=1)

train_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4))

history = train_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=35,
    callbacks=[SamplePreds(every=5), reduce_lr, early, ckpt],
)

print("\n✅ TRAINING DONE")
!ls -lh /kaggle/working | tail -n 20
print("✅ best weights:", ckpt_path)

Epoch 1/35


E0000 00:00:1771665100.860353      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/train_crnn_ctc_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1771665105.155912      62 cuda_dnn.cc:529] Loaded cuDNN version 91002


225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 845ms/step - loss: 19094.3633
🔎 Sample Predictions:
🖼️ 570283233ac74c17be3549631d0edb8e-0012-34.jpg
GT  : يطلبه أهالي كفر عين ونقبل اقتراحهم المعقول وذهابه إلى تحقيق البقرة ومنع محاكمة
PRED: 

🖼️ 53d5b986fc814540bc7640dbf97a7940-0009-06.jpg
GT  : و٧٧٧ فلسا وكم أتمنى أن لا أرجي ديوني وأدفعها قبل استحقاقها.
PRED: 

🖼️ f7420c97147f4256afc592d58375f38a-0021-47.jpg
GT  : العراق وتركيا وإيران وباكستان يحمل مطالب الدول الموالية واخشى أن يكون
PRED: 


Epoch 1: val_loss improved from inf to 18392.92188, saving model to /kaggle/working/best_final_raw.weights.h5
225/225 ━━━━━━━━━━━━━━━━━━━━ 217s 896ms/step - loss: 19070.4023 - val_loss: 18392.9219 - learning_rate: 3.0000e-04
Epoch 2/35
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 839ms/step - loss: 11555.2578
Epoch 2: val_loss improved from 18392.92188 to 12351.73730, saving model to /kaggle/working/best_final_raw.weights.h5
225/225 ━━━━━━━━━━━━━━━━━━━━ 199s 876ms/step - loss: 11555.1768 - val_loss: 12351.7373 - learning_rate: 

In [6]:
!ls -lh /kaggle/working | egrep "best_.*weights\.h5|\.keras|train_log"

-rw-r--r-- 1 root root  46M Feb 21 10:58 best_final_raw.weights.h5


In [7]:
import os, re, math
import numpy as np
import pandas as pd
import tensorflow as tf
from collections import defaultdict
from tqdm import tqdm

# =========================
# PATHS (حسب مساراتك)
# =========================
test_csv = "/kaggle/input/datasets/geleefinal/gelee-test/AR-MS.blind_test/blind_test.csv"
test_images_dir = "/kaggle/input/datasets/geleefinal/gelee-test/AR-MS.blind_test/images"

WEIGHTS_PATH = "/kaggle/working/best_final_raw.weights.h5"

# =========================
# sanity checks
# =========================
print("✅ weights exists?", os.path.exists(WEIGHTS_PATH), WEIGHTS_PATH)
print("✅ test_csv exists?", os.path.exists(test_csv), test_csv)
print("✅ test_images_dir exists?", os.path.exists(test_images_dir), test_images_dir)

# =========================
# load weights
# =========================
prediction_model.load_weights(WEIGHTS_PATH)
print("✅ weights loaded")

img_width, img_height = 1536, 64

def normalize_text(s):
    s = re.sub(r"\s+", " ", str(s)).strip()
    s = re.sub(r"\s+([،\.!\?؛:])", r"\1", s)
    # حذف تكرار عالي
    s = re.sub(r"(.)\1{3,}", r"\1\1", s)
    return s

def preprocess_test(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=1, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, [img_height, img_width])
    img = tf.image.flip_left_right(img)
    img = tf.transpose(img, perm=[1, 0, 2])
    return img

# =========================
# LM-lite (char 5-gram) من نصوص التدريب
# =========================
N = 5
train_texts = [normalize_text(t) for t in df["text"].astype(str).tolist()]

ng_counts = [defaultdict(int) for _ in range(N+1)]
ctx_counts = [defaultdict(int) for _ in range(N+1)]
char_set = set()

for t in train_texts:
    t2 = "⟨" + t + "⟩"
    for ch in t2:
        char_set.add(ch)
    for k in range(1, N+1):
        for i in range(len(t2)-k+1):
            ng = t2[i:i+k]
            ng_counts[k][ng] += 1
            if k > 1:
                ctx_counts[k][ng[:-1]] += 1

V = len(char_set) + 10
unigram_total = sum(ng_counts[1].values())

def lm_score(text):
    t = "⟨" + normalize_text(text) + "⟩"
    if len(t) < 2:
        return -50.0
    total = 0.0
    steps = 0
    for i in range(len(t)):
        for k in range(N, 0, -1):
            if i-k+1 < 0:
                continue
            ng = t[i-k+1:i+1]
            if k == 1:
                c = ng_counts[1].get(ng, 0)
                p = (c + 1.0) / (unigram_total + V)
                total += math.log(p)
                steps += 1
                break
            else:
                c = ng_counts[k].get(ng, 0)
                ctx = ng[:-1]
                cc = ctx_counts[k].get(ctx, 0)
                p = (c + 1.0) / (cc + V)
                total += math.log(p)
                steps += 1
                break
    return total / max(steps, 1)

# =========================
# lexicon bonus من كلمات التدريب
# =========================
train_words = set()
for t in train_texts:
    for w in re.findall(r"[^\s]+", t):
        if 2 <= len(w) <= 25:
            train_words.add(w)

def lex_bonus(text):
    ws = re.findall(r"[^\s]+", normalize_text(text))
    if not ws:
        return 0.0
    hit = sum(1 for w in ws if w in train_words)
    return hit / len(ws)

print("✅ train_words:", len(train_words))

# =========================
# CTC candidates
# =========================
vocab = tf.constant(char_to_num.get_vocabulary())
vocab_size = int(vocab.shape[0])

def decode_candidates(pred, beam=160, top_paths=15):
    ilen = np.ones(pred.shape[0]) * pred.shape[1]
    decoded, log_probs = tf.keras.backend.ctc_decode(
        pred, input_length=ilen, greedy=False,
        beam_width=beam, top_paths=top_paths
    )
    out=[]
    for k in range(top_paths):
        seq = decoded[k][0]
        seq = seq[seq != -1]
        seq = seq[seq < vocab_size]
        if seq.shape[0] == 0:
            txt = ""
        else:
            txt = tf.strings.reduce_join(tf.gather(vocab, seq)).numpy().decode("utf-8")
        out.append((normalize_text(txt), float(log_probs[0][k])))
    return out

def pick_best(cands, alpha, beta, length_pow=0.7):
    best_t, best_s = "", -1e18
    for t, lp in cands:
        L = max(len(t), 1)
        ctc = lp / (L ** length_pow)
        s = ctc + alpha * lm_score(t) + beta * lex_bonus(t)

        # عقوبات سريعة للنص الفاضي/القصير
        if len(t) < 3: s -= 6.0
        if len(t.split()) < 2: s -= 1.0

        if s > best_s:
            best_s, best_t = s, t
    return best_t

# =========================
# 3 configs (A/B/C)
# =========================
configs = [
    ("A", 0.35, 0.10, 0.70),  # balanced
    ("B", 0.45, 0.05, 0.70),  # stronger LM
    ("C", 0.55, 0.00, 0.75),  # stronger LM + length tweak
]

test_df = pd.read_csv(test_csv)
filenames = test_df["filename"].tolist()
all_preds = {tag: [] for tag,_,_,_ in configs}

for i, name in enumerate(tqdm(filenames, desc="Predicting (3 submissions)")):
    path = os.path.join(test_images_dir, name)
    img = preprocess_test(path)
    pred = prediction_model.predict(tf.expand_dims(img,0), verbose=0)

    cands = decode_candidates(pred, beam=160, top_paths=15)

    for tag, a, b, lpw in configs:
        txt = pick_best(cands, alpha=a, beta=b, length_pow=lpw)
        all_preds[tag].append(txt)

    if i < 3:
        print("\n" + "—"*90)
        print("File:", name)
        for t, lp in cands[:5]:
            print(" ", round(lp,3), "->", t[:120])
        for tag,a,b,lpw in configs:
            print(f"✅ {tag} chosen (a={a}, b={b}, pow={lpw}):", all_preds[tag][-1][:200])

# save 3 submissions
for tag,_,_,_ in configs:
    out = test_df.copy()
    out["text"] = all_preds[tag]
    out_path = f"submission_{tag}.csv"
    out.to_csv(out_path, index=False, encoding="utf-8-sig")
    print("✅ wrote:", out_path)

!ls -lh submission_*.csv

✅ weights exists? True /kaggle/working/best_final_raw.weights.h5
✅ test_csv exists? True /kaggle/input/datasets/geleefinal/gelee-test/AR-MS.blind_test/blind_test.csv
✅ test_images_dir exists? True /kaggle/input/datasets/geleefinal/gelee-test/AR-MS.blind_test/images
✅ weights loaded
✅ train_words: 45824


Predicting (3 submissions):   0%|          | 1/2671 [00:02<1:50:21,  2.48s/it]


——————————————————————————————————————————————————————————————————————————————————————————
File: 1958_p112_l0033.png
  -5.061 -> أنهم لا يوافقون على قوة دولية ولا يستبدون بالقوات الانكليزىي المراقبين دولين
  -5.061 -> أنهم لا يوافقون على قوة دولية ولا يستبدلون بالقوات الانكليزىي المراقبين دولين
  -5.155 -> أنهم لا يوافقون على قوة دولية ولا يستبدون بالقوات الانكليزي المراقبين دولين
  -5.156 -> أنهم لا يوافقون على قوة دولية ولا يستبدلون بالقوات الانكليزي المراقبين دولين
  -5.371 -> إنهم لا يوافقون على قوة دولية ولا يستبدون بالقوات الانكليزىي المراقبين دولين
✅ A chosen (a=0.35, b=0.1, pow=0.7): أنهم لا يوافقون على قوة دولية ولا يستبدون بالقوات الانكليزي المراقبين دولين
✅ B chosen (a=0.45, b=0.05, pow=0.7): أنهم لا يوافقون على قوة دولية ولا يستبدون بالقوات الانكليزي المراقبين دولين
✅ C chosen (a=0.55, b=0.0, pow=0.75): أنهم لا يوافقون على قوة دولية ولا يستبدون بالقوات الانكليزي المراقبين دولين


Predicting (3 submissions):   0%|          | 2/2671 [00:04<1:37:54,  2.20s/it]


——————————————————————————————————————————————————————————————————————————————————————————
File: 1964_p167_l0044.png
  -16.942 -> يوجذ في الشسعورينة ف٨ ذ معلمة اردنية تعلم في مدارب الأردن هذه السنة غير المرفوضات
  -17.103 -> يوجذ في الشسعورينة ٨ ذ معلمة اردنية تعلم في مدارب الأردن هذه السنة غير المرفوضات
  -17.126 -> يوجذ في الشسعورينة ك٨ ذ معلمة اردنية تعلم في مدارب الأردن هذه السنة غير المرفوضات
  -17.14 -> يوجذ في السعورينة ف٨ ذ معلمة اردنية تعلم في مدارب الأردن هذه السنة غير المرفوضات
  -17.245 -> يوجذ في الشسعورينة ف٨ ذ معلمة أردنية تعلم في مدارب الأردن هذه السنة غير المرفوضات
✅ A chosen (a=0.35, b=0.1, pow=0.7): يوجذ في السعورينة ف٨ ذ معلمة اردنية تعلم في مدارب الأردن هذه السنة غير المرفوضات
✅ B chosen (a=0.45, b=0.05, pow=0.7): يوجذ في السعورينة ف٨ ذ معلمة أردنية تعلم في مدارب الأردن هذه السنة غير المرفوضات
✅ C chosen (a=0.55, b=0.0, pow=0.75): يوجذ في السعورينة ف٨ ذ معلمة أردنية تعلم في مدارب الأردن هذه السنة غير المرفوضات


Predicting (3 submissions):   0%|          | 3/2671 [00:06<1:35:38,  2.15s/it]


——————————————————————————————————————————————————————————————————————————————————————————
File: 1956_p080_l0012.png
  -11.44 -> النواب استقالت وإذا طنا لتها تربعت في الحكم وباشرت أعماها ني ضرم ورضاتة أما أن الملك يطلع
  -11.655 -> النواب استقالت وإذا هنا لتها تربعت في الحكم وباشرت أعماها ني ضرم ورضاتة أما أن الملك يطلع
  -11.817 -> النواب استقالت وإذا طنا لتها تربعت في الحكم وباشرت أعماها ني ضرم ورضاتة إما أن الملك يطلع
  -11.999 -> النواب استقالت وإذا طنا لتها تربعت في الحكم وباشرت أعماها ني خرم ورضاتة أما أن الملك يطلع
  -12.036 -> النواب استقالت وإذا هنا لتها تربعت في الحكم وباشرت أعماها ني ضرم ورضاتة إما أن الملك يطلع
✅ A chosen (a=0.35, b=0.1, pow=0.7): النواب استقالت وإذا هنا لتها تربعت في الحكم وباشرت أعماها ني ضرم ورضاتة أما أن الملك يطلع
✅ B chosen (a=0.45, b=0.05, pow=0.7): النواب استقالت وإذا هنا لتها تربعت في الحكم وباشرت أعماها ني ضرم ورضاتة أما أن الملك يطلع
✅ C chosen (a=0.55, b=0.0, pow=0.75): النواب استقالت وإذا هنا لتها تربعت في الحكم وباشرت أعماها ني ضرم ورضاتة أما

Predicting (3 submissions): 100%|██████████| 2671/2671 [1:27:20<00:00,  1.96s/it]

✅ wrote: submission_A.csv
✅ wrote: submission_B.csv
✅ wrote: submission_C.csv
-rw-r--r-- 1 root root 397K Feb 21 12:35 submission_A.csv
-rw-r--r-- 1 root root 397K Feb 21 12:35 submission_B.csv
-rw-r--r-- 1 root root 396K Feb 21 12:35 submission_C.csv


In [8]:
len(train_words)

45824

In [9]:
import re
from collections import Counter, defaultdict
import pandas as pd
from tqdm import tqdm

# ===============================
# 1) اقرأ ملف السبميشن الحالي
# ===============================
SUB_IN  = "/kaggle/working/submission_C.csv"
SUB_OUT = "submission_symspell.csv"

sub = pd.read_csv(SUB_IN)
print("✅ Loaded submission:", sub.shape)

# ===============================
# 2) Arabic normalization
# ===============================
AR_NORM_MAP = str.maketrans({
    "أ":"ا", "إ":"ا", "آ":"ا",
    "ى":"ي",
    "ؤ":"و", "ئ":"ي",
    "ـ":"",   # remove tatweel
})

def normalize_word(w: str) -> str:
    w = w.strip()
    w = re.sub(r"[^\w\u0600-\u06FF]+", "", w)
    w = w.translate(AR_NORM_MAP)
    w = re.sub(r"(.)\1{3,}", r"\1\1", w)
    return w

def is_arabic_word(w: str) -> bool:
    return re.search(r"[\u0600-\u06FF]", w) is not None

# ===============================
# 3) بناء قاموس كلمات التدريب
# ===============================
train_texts = df["text"].astype(str).tolist()

word_counter = Counter()
for t in train_texts:
    t = re.sub(r"\s+", " ", t).strip()
    for w in t.split():
        nw = normalize_word(w)
        if 2 <= len(nw) <= 25 and is_arabic_word(nw):
            word_counter[nw] += 1

vocab = set(word_counter.keys())
print("✅ Vocab size:", len(vocab))

# ===============================
# 4) SymSpell deletes index
# ===============================
MAX_EDIT = 2

def deletes(word: str, max_edit=2):
    out = set()
    queue = {word}
    for _ in range(max_edit):
        newq = set()
        for w in queue:
            if len(w) <= 1:
                continue
            for i in range(len(w)):
                dw = w[:i] + w[i+1:]
                if dw not in out:
                    out.add(dw)
                    newq.add(dw)
        queue = newq
    return out

delete_index = defaultdict(list)

print("🔨 Building delete index...")
for w in tqdm(vocab):
    for d in deletes(w, MAX_EDIT):
        delete_index[d].append(w)

print("✅ Delete index built:", len(delete_index))

# ===============================
# 5) Edit distance محدود
# ===============================
def edit_distance(a: str, b: str, max_dist=2) -> int:
    if abs(len(a) - len(b)) > max_dist:
        return max_dist + 1

    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        row_min = cur[0]
        for j, cb in enumerate(b, 1):
            cost = 0 if ca == cb else 1
            cur.append(min(
                prev[j] + 1,
                cur[j-1] + 1,
                prev[j-1] + cost
            ))
            row_min = min(row_min, cur[-1])
        prev = cur
        if row_min > max_dist:
            return max_dist + 1
    return prev[-1]

def symspell_correct(word: str):
    if word in vocab or len(word) < 2:
        return word

    candidates = set()
    for d in deletes(word, MAX_EDIT):
        for cand in delete_index.get(d, []):
            candidates.add(cand)

    if not candidates:
        return word

    best = word
    best_dist = MAX_EDIT + 1
    best_freq = -1

    for c in candidates:
        dist = edit_distance(word, c, MAX_EDIT)
        if dist <= MAX_EDIT:
            freq = word_counter.get(c, 0)
            if (dist < best_dist) or (dist == best_dist and freq > best_freq):
                best = c
                best_dist = dist
                best_freq = freq

    return best

# ===============================
# 6) تطبيق التصحيح مع طباعة تقدم
# ===============================
total_words = 0
corrected_words = 0
preview_limit = 5
preview_count = 0

new_texts = []

print("\n🚀 Starting word-level correction...\n")

for i, row in tqdm(sub.iterrows(), total=len(sub), desc="🔧 Correcting"):
    original = str(row["text"])
    parts = re.findall(r"\w+|[^\w\s]", original, flags=re.UNICODE)

    new_parts = []

    for p in parts:
        if re.match(r"^\w+$", p, flags=re.UNICODE) and is_arabic_word(p):
            total_words += 1
            nw = normalize_word(p)

            if nw:
                cw = symspell_correct(nw)

                if cw != nw:
                    corrected_words += 1
                    if preview_count < preview_limit:
                        print("\n📝 Example correction:")
                        print("   BEFORE:", p)
                        print("   AFTER :", cw)
                        preview_count += 1

                new_parts.append(cw)
            else:
                new_parts.append(p)
        else:
            new_parts.append(p)

    sentence = " ".join(new_parts)
    sentence = re.sub(r"\s+([،\.!\?؛:])", r"\1", sentence)
    sentence = re.sub(r"\s+", " ", sentence).strip()

    new_texts.append(sentence)

    if (i+1) % 200 == 0:
        print(f"\n📊 {i+1}/{len(sub)} processed")
        print(f"   Words checked: {total_words}")
        print(f"   Words corrected: {corrected_words}")
        print(f"   Correction rate: {corrected_words/max(total_words,1):.4f}")

sub["text"] = new_texts

print("\n✅ Correction finished")
print("Total words:", total_words)
print("Corrected words:", corrected_words)
print("Final correction rate:", corrected_words/max(total_words,1))

# ===============================
# 7) حفظ الملف الجديد
# ===============================
sub.to_csv(SUB_OUT, index=False, encoding="utf-8-sig")
print("✅ Saved:", SUB_OUT)

✅ Loaded submission: (2671, 2)
✅ Vocab size: 41766
🔨 Building delete index...


100%|██████████| 41766/41766 [00:01<00:00, 25441.10it/s]


✅ Delete index built: 392449

🚀 Starting word-level correction...



🔧 Correcting:   0%|          | 5/2671 [00:00<00:56, 46.95it/s]


📝 Example correction:
   BEFORE: بالقوات
   AFTER : القوات

📝 Example correction:
   BEFORE: دولين
   AFTER : دولية

📝 Example correction:
   BEFORE: يوجذ
   AFTER : يوجد

📝 Example correction:
   BEFORE: السعورينة
   AFTER : السعودية

📝 Example correction:
   BEFORE: ف٨
   AFTER : في


...


📊 2400/2671 processed
   Words checked: 27177
   Words corrected: 7287
   Correction rate: 0.2681


🔧 Correcting:  98%|█████████▊| 2619/2671 [00:27<00:00, 128.86it/s]


📊 2600/2671 processed
   Words checked: 29248
   Words corrected: 7746
   Correction rate: 0.2648


🔧 Correcting: 100%|██████████| 2671/2671 [00:28<00:00, 94.47it/s] 


✅ Correction finished
Total words: 29909
Corrected words: 7876
Final correction rate: 0.2633321073924237
✅ Saved: submission_symspell.csv


In [10]:
import pandas as pd

# المسارات
old_sub_path = "/kaggle/input/datasets/geleefinal/top-reading/submission (8).csv"
new_sub_path = "/kaggle/working/submission_C.csv"  # عدلي إذا اسم ملفك الحالي مختلف

old_df = pd.read_csv(old_sub_path)
new_df = pd.read_csv(new_sub_path)

merged = old_df.merge(new_df, on="filename", suffixes=("_old", "_new"))

diff = merged[merged["text_old"] != merged["text_new"]]

print("Total samples:", len(merged))
print("Different samples:", len(diff))

print("\n🔎 First 15 differences:\n")
display(diff.head(15))

Total samples: 2671
Different samples: 2281

🔎 First 15 differences:



,filename,text_old,text_new
0,1958_p112_l0033.png,إنهم لا يوافقون على قوة دولية ولا يستبدلون بال...,أنهم لا يوافقون على قوة دولية ولا يستبدون بالق...
1,1964_p167_l0044.png,بوجد في السعودي ٧٠ معلمة أردنية تعلم في مدارس ...,يوجذ في السعورينة ف٨ ذ معلمة أردنية تعلم في مد...
2,1956_p080_l0012.png,النواب استقالت وإذا ل نالتمها ترعت في الحكم وب...,النواب استقالت وإذا هنا لتها تربعت في الحكم وب...
3,1960_p116_l0063.png,جستدقا بعرة فسيفته من مل تاحية وصوب ويشعراز بل...,جتقاونيترف يفته منها باحية وصو ويشعر أنه بع كل...
4,1962_p077_l0032.png,د احي سشوايدو المرعة الذبي شمو بر عدير برمشعل ...,حا و يدو:المرعة النربية توبر عد يرابر مشقل جم ...
5,1964b_p077_l0044.png,بوجد في السعودي ٧٠ معلمة أردنية تعلم في مدارس ...,يوجذ في السعورينة ف٨ ذ معلمة أردنية تعلم في مد...
6,1958_p012_l0029.png,وعنده ذهب تقد رقيمته بمبلغي٠.٠ دنيار خير الجني...,وعنده ذهب تقد رفيمته بمبلغ.٠ دنيار غير الجنيها...
7,1962_p173_l0055.png,(٣) الشاء محكمة عدل عليا اطلسية. وأقر المؤتمر ...,(٣) انشاء محكمة عدل عليا أطلسية. وأقر المؤتمر ...
8,1954_p068_l0020.png,إن أمام هته العاصفة أختالاتا ١(١) أما آن تهدا ...,إن أمام هذه العاصفة أتالاش ١(١٠ إما أن تهد ابن...
9,1960_p104_l0015.png,أة أمريكا تستورد ٦٠ب% من تاج كركوبا أو ماميادل...,ن أميريكا تستورد ٠% من أتاج سكركوا أو مايادل ٦...


In [11]:
prediction_model.load_weights("/kaggle/working/best_final_raw.weights.h5")

In [12]:
import inspect
print(inspect.getsource(preprocess_image))

def preprocess_image(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=1, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, [img_height, img_width])

    # RTL
    img = tf.image.flip_left_right(img)
    # time axis
    img = tf.transpose(img, perm=[1, 0, 2])
    return img



In [13]:
import os, re
import numpy as np
import pandas as pd
import tensorflow as tf
from tqdm import tqdm

# ========= Paths (حسب اللي عطيتيني) =========
test_csv    = "/kaggle/input/datasets/geleefinal/gelee-test/AR-MS.blind_test/blind_test.csv"
test_images = "/kaggle/input/datasets/geleefinal/gelee-test/AR-MS.blind_test/images"
out_path    = "/kaggle/working/submission.csv"

# ========= Image shape (لازم نفس التدريب) =========
img_width  = 1536
img_height = 64

# ========= Load weights =========
ckpt_path = "/kaggle/working/best_final_raw.weights.h5"   # عدّليه إذا اسم وزنك مختلف
prediction_model.load_weights(ckpt_path)
print("✅ loaded:", ckpt_path)

# ========= vocab from training lookup =========
vocab = tf.constant(char_to_num.get_vocabulary())
vocab_size = int(vocab.shape[0])
print("✅ vocab_size:", vocab_size)

# ========= EXACT SAME preprocess as training =========
def preprocess_image_test(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=1, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, [img_height, img_width])
    img = tf.image.flip_left_right(img)
    img = tf.transpose(img, perm=[1, 0, 2])
    return img

# ========= Light post (نفس فكرتك) =========
def light_post(s):
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"\s+([،\.!\?؛:])", r"\1", s)
    return s

# ========= Beam decode (raw) =========
BEAM_WIDTH = 80  # جرّبي 50/80/120 إذا تبين

def decode_beam(pred, beam_width=BEAM_WIDTH):
    ilen = np.ones(pred.shape[0]) * pred.shape[1]
    decoded, _ = tf.keras.backend.ctc_decode(
        pred, input_length=ilen,
        greedy=False, beam_width=beam_width, top_paths=1
    )
    seq = decoded[0][0]
    seq = seq[seq != -1]
    seq = seq[seq < vocab_size]
    if seq.shape[0] == 0:
        return ""
    return tf.strings.reduce_join(tf.gather(vocab, seq)).numpy().decode("utf-8")

# ========= Predict =========
test_df = pd.read_csv(test_csv)

preds = []
print("🚀 Predicting...")

for i, name in enumerate(tqdm(test_df["filename"].tolist(), desc=f"Predicting (beam={BEAM_WIDTH})")):
    path = os.path.join(test_images, name)
    img  = preprocess_image_test(path)
    pred = prediction_model.predict(tf.expand_dims(img, 0), verbose=0)

    txt = light_post(decode_beam(pred, BEAM_WIDTH))
    preds.append(txt)

    if i < 5:
        print("—"*90)
        print("File:", name)
        print("Pred:", txt[:200])

test_df["text"] = preds
test_df.to_csv(out_path, index=False, encoding="utf-8-sig")
print("✅ saved:", out_path)

✅ loaded: /kaggle/working/best_final_raw.weights.h5
✅ vocab_size: 138
🚀 Predicting...


Predicting (beam=80):   0%|          | 1/2671 [00:01<45:55,  1.03s/it]

——————————————————————————————————————————————————————————————————————————————————————————
File: 1958_p112_l0033.png
Pred: أنهم لا يوافقون على قوة دولية ولا يستبدون بالقوات الانكليزىي المراقبين دولين


Predicting (beam=80):   0%|          | 2/2671 [00:02<46:16,  1.04s/it]

——————————————————————————————————————————————————————————————————————————————————————————
File: 1964_p167_l0044.png
Pred: يوجذ في الشسعورينة ف٨ ذ معلمة اردنية تعلم في مدارب الأردن هذه السنة غير المرفوضات


...

——————————————————————————————————————————————————————————————————————————————————————————
File: 1960_p116_l0063.png
Pred: جتقاونيترف يفته منه بآحية وصو ويشعر أنه بع كل ما يشتهي


Predicting (beam=80):   0%|          | 5/2671 [00:05<45:34,  1.03s/it]

——————————————————————————————————————————————————————————————————————————————————————————
File: 1962_p077_l0032.png
Pred: حا و يدو:المرعة النربية توبر عد يرابر مشقل جم الاشبتين شقبه ويالو وديراسين


Predicting (beam=80): 100%|██████████| 2671/2671 [43:21<00:00,  1.03it/s]

✅ saved: /kaggle/working/submission.csv


In [14]:
import pandas as pd
import re

old_path = "/kaggle/input/datasets/geleefinal/top-reading/submission (8).csv"  # 0.12
new_path = "/kaggle/working/submission.csv"                                   # آخر مخرجات موديلك
out_path = "/kaggle/working/submission_ensemble.csv"

old_df = pd.read_csv(old_path)
new_df = pd.read_csv(new_path)

m = old_df.merge(new_df, on="filename", suffixes=("_old", "_new"))

arabic_re = re.compile(r"[\u0600-\u06FF]")
def quality_score(s: str) -> float:
    if not isinstance(s, str): s = ""
    s2 = s.strip()
    if len(s2) == 0: 
        return -1e9
    # نسبة الحروف العربية
    ar = len(arabic_re.findall(s2))
    score = 0.0
    score += ar * 1.0
    score += len(s2) * 0.05
    # عقوبات
    if len(s2) < 8: score -= 10
    if s2.count(" ") > len(s2) * 0.3: score -= 5   # مسافات مبالغ فيها
    if re.search(r"(.)\1\1\1", s2): score -= 3      # تكرار حرف كثير
    return score

chosen = []
changed = 0

for _, row in m.iterrows():
    o = row["text_old"]
    n = row["text_new"]

    so = quality_score(o)
    sn = quality_score(n)

    # قاعدة محافظة:
    # لا نبدّل إلا إذا "الجديد" واضح أنه أفضل جودة بكثير
    if sn > so + 12:
        chosen.append(n)
        changed += 1
    else:
        chosen.append(o)

m_out = pd.DataFrame({"filename": m["filename"], "text": chosen})
m_out.to_csv(out_path, index=False, encoding="utf-8-sig")

print("✅ saved:", out_path)
print("changed lines:", changed, "out of", len(m_out))
print(m_out.head(5))

✅ saved: /kaggle/working/submission_ensemble.csv
changed lines: 1 out of 2671
              filename                                               text
0  1958_p112_l0033.png  إنهم لا يوافقون على قوة دولية ولا يستبدلون بال...
1  1964_p167_l0044.png  بوجد في السعودي ٧٠ معلمة أردنية تعلم في مدارس ...
2  1956_p080_l0012.png  النواب استقالت وإذا ل نالتمها ترعت في الحكم وب...
3  1960_p116_l0063.png  جستدقا بعرة فسيفته من مل تاحية وصوب ويشعراز بل...
4  1962_p077_l0032.png  د احي سشوايدو المرعة الذبي شمو بر عدير برمشعل ...


In [15]:
import pandas as pd
import re

old_path = "/kaggle/input/datasets/geleefinal/top-reading/submission (8).csv"
new_path = "/kaggle/working/submission.csv"
out_path = "/kaggle/working/submission_ensemble_targeted.csv"

old_df = pd.read_csv(old_path)
new_df = pd.read_csv(new_path)
m = old_df.merge(new_df, on="filename", suffixes=("_old","_new"))

arabic_re = re.compile(r"[\u0600-\u06FF]")

def looks_bad(s: str) -> bool:
    if not isinstance(s, str): s = ""
    s = s.strip()
    if len(s) < 8:  # قصير جدًا
        return True
    ar = len(arabic_re.findall(s))
    if ar < 3:      # تقريبًا ما فيه عربي
        return True
    if re.search(r"(.)\1\1\1", s):  # تكرار حرف 4 مرات
        return True
    if s.count(" ") > 20:           # مسافات شاذة
        return True
    return False

changed = 0
chosen = []
for _, r in m.iterrows():
    o, n = r["text_old"], r["text_new"]
    if looks_bad(o) and isinstance(n, str) and len(n.strip()) >= 8:
        chosen.append(n)
        changed += 1
    else:
        chosen.append(o)

out = pd.DataFrame({"filename": m["filename"], "text": chosen})
out.to_csv(out_path, index=False, encoding="utf-8-sig")
print("✅ saved:", out_path)
print("changed lines:", changed)
print(out.head(5))

✅ saved: /kaggle/working/submission_ensemble_targeted.csv
changed lines: 19
              filename                                               text
0  1958_p112_l0033.png  إنهم لا يوافقون على قوة دولية ولا يستبدلون بال...
1  1964_p167_l0044.png  بوجد في السعودي ٧٠ معلمة أردنية تعلم في مدارس ...
2  1956_p080_l0012.png  النواب استقالت وإذا ل نالتمها ترعت في الحكم وب...
3  1960_p116_l0063.png  جستدقا بعرة فسيفته من مل تاحية وصوب ويشعراز بل...
4  1962_p077_l0032.png  د احي سشوايدو المرعة الذبي شمو بر عدير برمشعل ...


In [16]:
import pandas as pd
import re

old_path = "/kaggle/input/datasets/geleefinal/top-reading/submission (8).csv"
new_path = "/kaggle/working/submission.csv"
out_path = "/kaggle/working/submission_attack_numbers.csv"

old_df = pd.read_csv(old_path)
new_df = pd.read_csv(new_path)
m = old_df.merge(new_df, on="filename", suffixes=("_old","_new"))

digit_re = re.compile(r"[0-9٠-٩]")

def count_digits(s):
    return len(digit_re.findall(str(s)))

def longest_digit_seq(s):
    nums = re.findall(r"[0-9٠-٩]+", str(s))
    return max([len(x) for x in nums], default=0)

changed = 0
chosen = []

for _, r in m.iterrows():
    o = r["text_old"]
    n = r["text_new"]

    co = count_digits(o)
    cn = count_digits(n)

    lo = longest_digit_seq(o)
    ln = longest_digit_seq(n)

    replace = False

    # إذا الجديد فيه أرقام أكثر
    if cn > co:
        replace = True

    # إذا أطول رقم في الجديد أطول
    if ln > lo:
        replace = True

    # إذا الجديد فيه % والقديم لا
    if "%" in str(n) and "%" not in str(o):
        replace = True

    if replace:
        chosen.append(n)
        changed += 1
    else:
        chosen.append(o)

out = pd.DataFrame({"filename": m["filename"], "text": chosen})
out.to_csv(out_path, index=False, encoding="utf-8-sig")

print("✅ saved:", out_path)
print("changed lines:", changed)

✅ saved: /kaggle/working/submission_attack_numbers.csv
changed lines: 90


In [17]:
import pandas as pd
import re
from difflib import SequenceMatcher

old_path = "/kaggle/input/datasets/geleefinal/top-reading/submission (8).csv"
new_path = "/kaggle/working/submission.csv"
out_path = "/kaggle/working/submission_smart_attack.csv"

old_df = pd.read_csv(old_path)
new_df = pd.read_csv(new_path)
m = old_df.merge(new_df, on="filename", suffixes=("_old","_new"))

arabic_re = re.compile(r"[\u0600-\u06FF]")
digit_re = re.compile(r"[0-9٠-٩]")

def arabic_ratio(s):
    if not isinstance(s,str) or len(s)==0:
        return 0
    return len(arabic_re.findall(s)) / len(s)

def digit_count(s):
    return len(digit_re.findall(str(s)))

def similar(a,b):
    return SequenceMatcher(None, str(a), str(b)).ratio()

changed = 0
chosen = []

for _, r in m.iterrows():
    o = str(r["text_old"])
    n = str(r["text_new"])

    # تشابه عالي = فرق بسيط
    sim = similar(o,n)

    replace = False

    if sim > 0.80:  # قريبين جدًا
        # الجديد أطول قليلاً
        if len(n) > len(o) and len(n) - len(o) < 6:
            # نسبة العربي أعلى
            if arabic_ratio(n) >= arabic_ratio(o):
                # ما خسر أرقام
                if digit_count(n) >= digit_count(o):
                    replace = True

    if replace:
        chosen.append(n)
        changed += 1
    else:
        chosen.append(o)

out = pd.DataFrame({"filename": m["filename"], "text": chosen})
out.to_csv(out_path, index=False, encoding="utf-8-sig")

print("✅ saved:", out_path)
print("changed lines:", changed)

✅ saved: /kaggle/working/submission_smart_attack.csv
changed lines: 359


In [18]:
import pandas as pd
import re
from difflib import SequenceMatcher

old_path = "/kaggle/input/datasets/geleefinal/top-reading/submission (8).csv"
new_path = "/kaggle/working/submission.csv"
out_path = "/kaggle/working/submission_ultra_pp.csv"

old_df = pd.read_csv(old_path)
new_df = pd.read_csv(new_path)
m = old_df.merge(new_df, on="filename", suffixes=("_old","_new"))

digit_re  = re.compile(r"[0-9٠-٩]")
arabic_re = re.compile(r"[\u0600-\u06FF]")

def similar(a, b):
    return SequenceMatcher(None, a, b).ratio()

def digit_count(s):
    return len(digit_re.findall(s))

def arabic_ratio(s):
    s = str(s)
    if len(s) == 0:
        return 0.0
    return len(arabic_re.findall(s)) / len(s)

def has_weird_repeats(s):
    return re.search(r"(.)\1\1\1", s) is not None

def has_weird_spaces(s):
    return ("  " in s) or (s.count(" ") > 25)

def char_diff_approx(a, b):
    sm = SequenceMatcher(None, a, b)
    diff = 0
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            continue
        diff += max(i2 - i1, j2 - j1)
    return diff

changed = 0
chosen = []

for _, r in m.iterrows():
    o = str(r["text_old"]).strip()
    n = str(r["text_new"]).strip()

    # حمايات سريعة
    if len(n) < 10 or has_weird_repeats(n) or has_weird_spaces(n):
        chosen.append(o); continue

    # لازم نفس عدد الأرقام (Ultra++)
    if digit_count(n) != digit_count(o):
        chosen.append(o); continue

    # تشابه شديد جدًا
    sim = similar(o, n)
    if sim < 0.965:
        chosen.append(o); continue

    # فرق طول = حرف واحد فقط (إكمال حرف)
    dlen = len(n) - len(o)
    if dlen != 1:
        chosen.append(o); continue

    # تغييرات قليلة جدًا
    diff = char_diff_approx(o, n)
    if diff > 2:
        chosen.append(o); continue

    # لا نسمح أن العربية تقل
    if arabic_ratio(n) + 0.002 < arabic_ratio(o):
        chosen.append(o); continue

    # ✅ استبدال
    chosen.append(n)
    changed += 1

out = pd.DataFrame({"filename": m["filename"], "text": chosen})
out.to_csv(out_path, index=False, encoding="utf-8-sig")

print("✅ saved:", out_path)
print("changed lines:", changed)
print(out.head(3))

✅ saved: /kaggle/working/submission_ultra_pp.csv
changed lines: 90
              filename                                               text
0  1958_p112_l0033.png  إنهم لا يوافقون على قوة دولية ولا يستبدلون بال...
1  1964_p167_l0044.png  بوجد في السعودي ٧٠ معلمة أردنية تعلم في مدارس ...
2  1956_p080_l0012.png  النواب استقالت وإذا ل نالتمها ترعت في الحكم وب...


In [19]:
import pandas as pd
import re
from difflib import SequenceMatcher

old_path = "/kaggle/input/datasets/geleefinal/top-reading/submission (8).csv"  # أفضل مرجع
new_path = "/kaggle/working/submission.csv"  # ناتجك الحالي
out_low  = "/kaggle/working/sub_selective_LOW.csv"
out_mid  = "/kaggle/working/sub_selective_MID.csv"
out_high = "/kaggle/working/sub_selective_HIGH.csv"

old_df = pd.read_csv(old_path)
new_df = pd.read_csv(new_path)
m = old_df.merge(new_df, on="filename", suffixes=("_old","_new"))

arabic_re = re.compile(r"[\u0600-\u06FF]")
digit_re  = re.compile(r"[0-9٠-٩]")

def ar_ratio(s):
    s = str(s)
    if len(s.strip()) == 0: return 0.0
    return len(arabic_re.findall(s)) / len(s)

def word_count(s):
    s = str(s).strip()
    if not s: return 0
    return len(s.split())

def digit_count(s):
    return len(digit_re.findall(str(s)))

def weird_penalty(s):
    s = str(s)
    pen = 0.0
    # تكرار نفس الحرف كثير
    if re.search(r"(.)\1\1\1", s): pen += 2.0
    # مسافات كثيرة
    if "  " in s: pen += 1.0
    # حروف قليلة جدًا مقارنة بالطول (هذيانة)
    if ar_ratio(s) < 0.55: pen += 2.0
    # كلمات قليلة جدًا
    if word_count(s) <= 2: pen += 1.5
    return pen

def score(s):
    s = str(s)
    # مكافأة العربية + مكافأة طول معتدل + عقوبات للهذيان
    return (ar_ratio(s) * 3.0) + (min(len(s), 120) / 120.0) + (min(word_count(s), 18) / 18.0) - weird_penalty(s)

def sim(a,b):
    return SequenceMatcher(None, str(a), str(b)).ratio()

def decide(o, n, mode="LOW"):
    o = str(o).strip()
    n = str(n).strip()

    # لا تخسري الأرقام
    if digit_count(n) < digit_count(o):
        return o, False

    # لا تسمحي بجديد فاضي/قصير جدًا
    if len(n) < 12:
        return o, False

    # لازم يكون في تشابه منطقي (ما نبي قفزة لهذيان)
    if sim(o,n) < 0.55:
        return o, False

    so, sn = score(o), score(n)
    diff = sn - so

    # عتبات مختلفة
    if mode == "LOW":
        thr = 0.55   # شديد المحافظة
    elif mode == "MID":
        thr = 0.35
    else:  # HIGH
        thr = 0.20

    # كمان نطلب أن العربية في الجديد ما تقل
    if ar_ratio(n) + 0.02 < ar_ratio(o):
        return o, False

    if diff > thr:
        return n, True
    return o, False

def build(mode, out_path):
    chosen=[]
    changed=0
    for _, r in m.iterrows():
        o, n = r["text_old"], r["text_new"]
        t, ch = decide(o, n, mode=mode)
        chosen.append(t)
        changed += int(ch)

    out = pd.DataFrame({"filename": m["filename"], "text": chosen})
    out.to_csv(out_path, index=False, encoding="utf-8-sig")
    return changed

c_low  = build("LOW",  out_low)
c_mid  = build("MID",  out_mid)
c_high = build("HIGH", out_high)

print("✅ saved:")
print("LOW :", out_low,  "changed:", c_low)
print("MID :", out_mid,  "changed:", c_mid)
print("HIGH:", out_high, "changed:", c_high)

✅ saved:
LOW : /kaggle/working/sub_selective_LOW.csv changed: 0
MID : /kaggle/working/sub_selective_MID.csv changed: 0
HIGH: /kaggle/working/sub_selective_HIGH.csv changed: 4


In [20]:
import pandas as pd
import re
from collections import defaultdict

# ====== 1) اقرأ السبميشن الحالي ======
in_sub  = "/kaggle/input/datasets/geleefinal/top-reading/submission (8).csv"   # عدّلي لو تبين (مثلاً submission (8).csv)
out_sub = "/kaggle/working/sub_lex_ed1.csv"

df = pd.read_csv(in_sub)

# ====== 2) تأكد أن train_words موجودة ======
# لازم تكون set من كلمات التدريب
assert "train_words" in globals(), "❌ train_words غير موجودة. سويها set قبل."
if not isinstance(train_words, set):
    train_words = set(train_words)

print("✅ train_words size:", len(train_words))
print("✅ rows:", len(df))

# ====== 3) أدوات مساعدة ======
arabic_letter_re = re.compile(r"^[\u0600-\u06FF]+$")  # كلمة عربية فقط
has_digit_re = re.compile(r"[0-9٠-٩]")

def one_sub_neighbors(w):
    # كل الكلمات الناتجة عن تبديل حرف واحد
    letters = "ابتثجحخدذرزسشصضطظعغفقكلمنهويىةؤئأإآء"
    out = []
    for i, ch in enumerate(w):
        for r in letters:
            if r != ch:
                out.append(w[:i] + r + w[i+1:])
    return out

def normalize_token(tok):
    # تطبيع خفيف جدًا بدون ما نخرب (مهم)
    tok = tok.strip()
    tok = tok.replace("ـ", "")  # كَشيدة
    return tok

def split_keep_punct(text):
    # نفصل الكلمات ونخلي علامات الترقيم لحالها
    # مثال: "المعقول." -> ["المعقول", "."]
    return re.findall(r"[\u0600-\u06FF]+|[0-9٠-٩]+|[^\s]", text)

def join_tokens(tokens):
    # نرجع نص طبيعي: نلصق علامات الترقيم بدون مسافة قبلها
    out=[]
    for t in tokens:
        if not out:
            out.append(t); continue
        if re.match(r"^[\.\,\!\?\:\;\،\؛\)\]\}]+$", t):
            out[-1] = out[-1] + t
        elif re.match(r"^[\(\[\{]+$", t):
            out.append(t)
        else:
            # مسافة قبل الكلمات/الأرقام
            out.append(" " + t)
    return "".join(out)

# ====== 4) بنينا فهرس حسب الطول لتسريع البحث ======
by_len = defaultdict(list)
for w in train_words:
    by_len[len(w)].append(w)

# ====== 5) التصحيح ======
changed_words = 0
changed_lines = 0

def correct_word(w):
    w0 = normalize_token(w)

    # لا نلمس الأرقام أو الكلمات القصيرة جدًا
    if has_digit_re.search(w0): 
        return w0, False
    if len(w0) < 4:
        return w0, False
    if not arabic_letter_re.match(w0):
        return w0, False

    # إذا الكلمة أصلًا موجودة
    if w0 in train_words:
        return w0, False

    # جرّب تبديل حرف واحد فقط (edit distance ~ 1 substitution)
    # نختار أول تطابق (أو تقدرين تغييره لأفضل اختيار)
    for cand in one_sub_neighbors(w0):
        if cand in train_words:
            return cand, True

    return w0, False

# طباعة تقدم
for i in range(len(df)):
    text = str(df.loc[i, "text"]) if "text" in df.columns else str(df.loc[i, "prediction"])
    toks = split_keep_punct(text)

    new_toks=[]
    line_changed=False

    for t in toks:
        ct, ch = correct_word(t)
        new_toks.append(ct)
        if ch:
            changed_words += 1
            line_changed = True

    if line_changed:
        changed_lines += 1

    df.loc[i, "text"] = join_tokens(new_toks)

    if i < 3:  # عينات أول 3
        print("—"*80)
        print("BEFORE:", text[:160])
        print("AFTER :", df.loc[i, "text"][:160])

print("✅ changed_lines:", changed_lines)
print("✅ changed_words:", changed_words)

df.to_csv(out_sub, index=False, encoding="utf-8-sig")
print("✅ saved:", out_sub)

✅ train_words size: 45824
✅ rows: 2671
————————————————————————————————————————————————————————————————————————————————
BEFORE: إنهم لا يوافقون على قوة دولية ولا يستبدلون بالقوات الانكليزي المراقبين دولين
AFTER : إنهم لا يوافقون على قوة دولية ولا يستبدلون والقوات الانكليزي المراقبين دولية
————————————————————————————————————————————————————————————————————————————————
BEFORE: بوجد في السعودي ٧٠ معلمة أردنية تعلم في مدارس الأردن هذه السنة غير المرفوضات
AFTER : توجد في السعودي ٧٠ معلمة أردنية تعلم في مدارس الأردن هذه السنة غير المرفوضات
————————————————————————————————————————————————————————————————————————————————
BEFORE: النواب استقالت وإذا ل نالتمها ترعت في الحكم وبأرشرت اعمالها في شرم ورضاتة أما أن الملك يطلع
AFTER : النواب استقالت وإذا ل نالتمها شرعت في الحكم وبأرشرت أعمالها في شرم ورضاتة أما أن الملك يطلع
✅ changed_lines: 1934
✅ changed_words: 4681
✅ saved: /kaggle/working/sub_lex_ed1.csv


In [21]:
import pandas as pd
import re

best_path  = "/kaggle/input/datasets/geleefinal/top-reading/submission (8).csv"
ultra_path = "/kaggle/working/submission_ultra_pp.csv"
out_path   = "/kaggle/working/sub_surgical_merge.csv"

df_best  = pd.read_csv(best_path)
df_ultra = pd.read_csv(ultra_path)

assert len(df_best) == len(df_ultra)

# تأكد أن train_words موجودة
assert "train_words" in globals()
if not isinstance(train_words, set):
    train_words = set(train_words)

has_digit = re.compile(r"[0-9٠-٩]")

def tokenize(t):
    return t.split()

changed_words = 0
changed_lines = 0

new_texts = []

for i in range(len(df_best)):

    text_best  = str(df_best.loc[i, "text"])
    text_ultra = str(df_ultra.loc[i, "text"])

    words_best  = tokenize(text_best)
    words_ultra = tokenize(text_ultra)

    if len(words_best) != len(words_ultra):
        new_texts.append(text_best)
        continue

    line_changed = False
    merged = []

    for w_best, w_ultra in zip(words_best, words_ultra):

        # شروط الأمان
        if (
            w_best != w_ultra
            and w_ultra in train_words
            and w_best not in train_words
            and not has_digit.search(w_best)
            and abs(len(w_best) - len(w_ultra)) <= 1
        ):
            merged.append(w_ultra)
            changed_words += 1
            line_changed = True
        else:
            merged.append(w_best)

    if line_changed:
        changed_lines += 1

    new_texts.append(" ".join(merged))

df_best["text"] = new_texts
df_best.to_csv(out_path, index=False, encoding="utf-8-sig")

print("✅ changed_lines:", changed_lines)
print("✅ changed_words:", changed_words)
print("✅ saved:", out_path)

✅ changed_lines: 10
✅ changed_words: 11
✅ saved: /kaggle/working/sub_surgical_merge.csv


In [22]:
import pandas as pd

best_path  = "/kaggle/input/datasets/geleefinal/top-reading/submission (8).csv"
ultra_path = "/kaggle/working/submission_ultra_pp.csv"
out_path   = "/kaggle/working/sub_surgical_merge.csv"

df_best  = pd.read_csv(best_path)
df_ultra = pd.read_csv(ultra_path)
df_out   = pd.read_csv(out_path)

diffs = []
for i in range(len(df_best)):
    b = str(df_best.loc[i,"text"])
    o = str(df_out.loc[i,"text"])
    if b != o:
        diffs.append((df_best.loc[i,"filename"], b[:180], o[:180]))

print("diff count:", len(diffs))
for fn, b, o in diffs[:20]:
    print("—"*80)
    print(fn)
    print("BEST :", b)
    print("MERGE:", o)

diff count: 10
————————————————————————————————————————————————————————————————————————————————
1965_p062_l0057.png
BEST : العهيونية والاستعمار ونود أن نقرن القول المنتج بالعمل اكمفيد والافماهى الفائدة من الاجتاعات
MERGE: العهيونية والاستعمار ونود أن نقرن القول المنتج بالعمل المفيد والافماهى الفائدة من الاجتاعات
————————————————————————————————————————————————————————————————————————————————
732294a1c8ef4d55ad023c877c098e26-0013-13.jpg
BEST : الجزائر بأن خيفر اختل هذه الأموال فإن صح هذا وعارض فيه خيفر يكون لصاخائنا
MERGE: الجزائر بأن خيضر اختل هذه الأموال فإن صح هذا وعارض فيه خيفر يكون لصاخائنا
————————————————————————————————————————————————————————————————————————————————
35df11aa660142d19d17069641aacf0f-0009-29.jpg
BEST : البارزة ولا اتكلف بالكتابة وليتني جمع بين الخطتين وابدى رأي واضحا.
MERGE: البارزة ولا اتكلف بالكتابة وليتني جمع بين الخطتين وأبدى رأي واضحا.
————————————————————————————————————————————————————————————————————————————————
1b363a3319eb499aab16e5460d6115ed-0018-25.jp

In [23]:
import os, re, math
import pandas as pd
from collections import Counter, defaultdict

# ====== paths (عدّلي لو تغيّر شيء) ======
train_csv   = "/kaggle/input/datasets/geleefinal/gelee-nakba/AR-MS.train/annotations.csv"

# ====== basic Arabic cleanup for SCORING only (لا نغير النص النهائي) ======
AR_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")  # حركات/تشكيل
TATWEEL = "\u0640"

def norm_for_scoring(s: str) -> str:
    s = str(s)
    s = s.replace(TATWEEL, "")
    s = AR_DIACRITICS.sub("", s)

    # توحيد أشكال شائعة (للسكور فقط)
    s = s.replace("أ","ا").replace("إ","ا").replace("آ","ا")
    s = s.replace("ى","ي")
    s = s.replace("ة","ه")  # بعض الموديلات تخربطها، للتقارب في السكور فقط
    s = s.replace("ؤ","و").replace("ئ","ي")

    # مسافات/ترقيم
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"\s+([،\.!\?؛:])", r"\1", s)
    return s

def word_tokenize(s: str):
    # نفصل الكلمات العربية/الأرقام، ونترك الترقيم كفاصل
    s = norm_for_scoring(s)
    s = re.sub(r"[\"“”‘’\(\)\[\]\{\}]", " ", s)
    s = re.sub(r"[،؛:!\?\.]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    if not s:
        return []
    return s.split()

# ====== build LM counts ======
df = pd.read_csv(train_csv)
df = df[df["text"].notna()]
texts = df["text"].astype(str).tolist()

uni = Counter()
bi  = Counter()
total_uni = 0

for t in texts:
    toks = ["<s>"] + word_tokenize(t) + ["</s>"]
    if len(toks) < 2:
        continue
    for w in toks[1:]:
        uni[w] += 1
        total_uni += 1
    for a, b in zip(toks[:-1], toks[1:]):
        bi[(a,b)] += 1

V = len(uni)
print("✅ train lines:", len(texts))
print("✅ Vocab size (words):", V)
print("✅ Bigram types:", len(bi))

✅ train lines: 15962
✅ Vocab size (words): 41271
✅ Bigram types: 131579


In [24]:
import pandas as pd
import math, re

# ====== your 2 submissions ======
subA_path = "/kaggle/input/datasets/geleefinal/top-reading/submission (8).csv"   # الأفضل
subB_path = "/kaggle/working/submission_ultra_pp.csv"                           # الثاني

testA = pd.read_csv(subA_path)
testB = pd.read_csv(subB_path)

# توحيد الأعمدة
for df_ in [testA, testB]:
    if "filename" not in df_.columns and "image" in df_.columns:
        df_.rename(columns={"image":"filename"}, inplace=True)
    if "text" not in df_.columns:
        raise ValueError("Missing text column in one of the submissions")

merged = testA[["filename","text"]].merge(
    testB[["filename","text"]],
    on="filename",
    how="inner",
    suffixes=("_A","_B")
)

print("✅ merged rows:", merged.shape[0])

def lm_score_text(text: str) -> float:
    toks = ["<s>"] + word_tokenize(text) + ["</s>"]
    if len(toks) < 2:
        return -1e9
    # add-k smoothing
    k = 0.1
    score = 0.0
    for a, b in zip(toks[:-1], toks[1:]):
        c_ab = bi.get((a,b), 0)
        c_a  = uni.get(a, 0)
        prob = (c_ab + k) / (c_a + k*V + 1e-9)
        score += math.log(prob + 1e-12)
    # طول-نورم بسيط عشان ما يفضّل القصير جدًا
    score /= max(1, (len(toks)-1)) ** 0.70
    return score

# إشارات بسيطة لصحة الأرقام (بدون “تصحيح” قهري)
arabic_indic = set(list("٠١٢٣٤٥٦٧٨٩"))
latin_digits = set(list("0123456789"))

def digit_quality(s: str) -> int:
    s = str(s)
    has_ar = any(ch in arabic_indic for ch in s)
    has_la = any(ch in latin_digits for ch in s)
    # نفضّل وجود أرقام عربية-هندية إذا وُجدت أرقام، لأن الداتا غالبًا كذلك
    if has_ar and not has_la: return 2
    if has_ar and has_la:     return 1
    if has_la and not has_ar: return 0
    return 1  # لا أرقام أساسًا

def choose_best(a: str, b: str):
    sa = lm_score_text(a)
    sb = lm_score_text(b)

    # boost بسيط لجودة الأرقام
    sa += 0.15 * digit_quality(a)
    sb += 0.15 * digit_quality(b)

    # كسر التعادل: لو واحد فاضي/قصير جدًا نخليه الثاني
    if len(str(a).strip()) < 3 and len(str(b).strip()) >= 3:
        return b, sa, sb
    if len(str(b).strip()) < 3 and len(str(a).strip()) >= 3:
        return a, sa, sb

    return (a if sa >= sb else b), sa, sb

chosen = []
pickA = 0
pickB = 0

for i, row in enumerate(merged.itertuples(index=False), start=1):
    a = row.text_A
    b = row.text_B
    best, sa, sb = choose_best(a, b)
    chosen.append(best)
    if best == a: pickA += 1
    else: pickB += 1

    if i <= 3 or i % 250 == 0:
        print("—"*80)
        print("i:", i, "| file:", row.filename)
        print("A:", str(a)[:140])
        print("B:", str(b)[:140])
        print(f"scoreA={sa:.3f}  scoreB={sb:.3f}  -> chosen:", str(best)[:140])

print("\n✅ picked A:", pickA, " | picked B:", pickB)
merged["text"] = chosen

✅ merged rows: 2671
————————————————————————————————————————————————————————————————————————————————
i: 1 | file: 1958_p112_l0033.png
A: إنهم لا يوافقون على قوة دولية ولا يستبدلون بالقوات الانكليزي المراقبين دولين
B: إنهم لا يوافقون على قوة دولية ولا يستبدلون بالقوات الانكليزي المراقبين دولين
scoreA=-20.571  scoreB=-20.571  -> chosen: إنهم لا يوافقون على قوة دولية ولا يستبدلون بالقوات الانكليزي المراقبين دولين
————————————————————————————————————————————————————————————————————————————————
i: 2 | file: 1964_p167_l0044.png
A: بوجد في السعودي ٧٠ معلمة أردنية تعلم في مدارس الأردن هذه السنة غير المرفوضات
B: بوجد في السعودي ٧٠ معلمة أردنية تعلم في مدارس الأردن هذه السنة غير المرفوضات
scoreA=-22.195  scoreB=-22.195  -> chosen: بوجد في السعودي ٧٠ معلمة أردنية تعلم في مدارس الأردن هذه السنة غير المرفوضات
————————————————————————————————————————————————————————————————————————————————
i: 3 | file: 1956_p080_l0012.png
A: النواب استقالت وإذا ل نالتمها ترعت في الحكم وبأرشرت اعمالها في شرم ورضاتة أ

In [25]:
out_path = "/kaggle/working/submission_ensemble_AB.csv"
merged[["filename","text"]].to_csv(out_path, index=False, encoding="utf-8-sig")

print("✅ saved:", out_path)
print("✅ preview:")
print(merged[["filename","text"]].head(10))

✅ saved: /kaggle/working/submission_ensemble_AB.csv
✅ preview:
               filename                                               text
0   1958_p112_l0033.png  إنهم لا يوافقون على قوة دولية ولا يستبدلون بال...
1   1964_p167_l0044.png  بوجد في السعودي ٧٠ معلمة أردنية تعلم في مدارس ...
2   1956_p080_l0012.png  النواب استقالت وإذا ل نالتمها ترعت في الحكم وب...
3   1960_p116_l0063.png  جستدقا بعرة فسيفته من مل تاحية وصوب ويشعراز بل...
4   1962_p077_l0032.png  د احي سشوايدو المرعة الذبي شمو بر عدير برمشعل ...
5  1964b_p077_l0044.png  بوجد في السعودي ٧٠ معلمة أردنية تعلم في مدارس ...
6   1958_p012_l0029.png  وعنده ذهب تقد رقيمته بمبلغي٠.٠ دنيار خير الجني...
7   1962_p173_l0055.png  (٣) الشاء محكمة عدل عليا اطلسية. وأقر المؤتمر ...
8   1954_p068_l0020.png  إن أمام هته العاصفة أختالاتا ١(١) أما آن تهدا ...
9   1960_p104_l0015.png  أة أمريكا تستورد ٦٠ب% من تاج كركوبا أو ماميادل...


In [26]:
import os, re, math
import pandas as pd
from collections import Counter

# ===================== PATHS =====================
train_csv = "/kaggle/input/datasets/geleefinal/gelee-nakba/AR-MS.train/annotations.csv"

SUB_PATHS = [
    "/kaggle/input/datasets/geleefinal/top-reading/submission (8).csv",              # A (أفضل)
    "/kaggle/input/datasets/geleefinal/best-2/Final_Submission_Rank1.csv",          # B
    "/kaggle/input/datasets/geleefinal/best-3/Final_Submission_Gold_V5.csv",        # C
    "/kaggle/input/datasets/geleefinal/best-4/LAST.csv",                            # D
    "/kaggle/input/datasets/geleefinal/best-5/submission (10).csv",                 # E
]

print("✅ Using submissions:")
for i,p in enumerate(SUB_PATHS,1):
    print(i, p)

# ===================== NORMALIZATION =====================
AR_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")  # تشكيل
TATWEEL = "\u0640"

def norm_for_match(s: str) -> str:
    """Normalization for matching/consensus (لا نغير المعنى كثير)."""
    s = str(s)
    s = s.replace(TATWEEL, "")
    s = AR_DIACRITICS.sub("", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def norm_for_lm(s: str) -> str:
    """Normalization for LM scoring (أقوى شوي للتهجئة)."""
    s = norm_for_match(s)
    s = s.replace("أ","ا").replace("إ","ا").replace("آ","ا")
    s = s.replace("ى","ي")
    s = s.replace("ؤ","و").replace("ئ","ي")
    return s

def word_tokenize(s: str):
    s = norm_for_lm(s)
    s = re.sub(r"[\"“”‘’\(\)\[\]\{\}]", " ", s)
    s = re.sub(r"[،؛:!\?\.]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s.split() if s else []

# ===================== LIGHT POST (final output safety) =====================
def light_post(s: str) -> str:
    s = str(s)
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"\s+([،\.!\?؛:])", r"\1", s)
    return s

# ===================== LOAD SUBS =====================
def load_sub(path: str) -> pd.DataFrame:
    d = pd.read_csv(path)
    if "filename" not in d.columns and "image" in d.columns:
        d = d.rename(columns={"image":"filename"})
    assert "filename" in d.columns and "text" in d.columns, f"Bad columns in {path}: {d.columns}"
    d["filename"] = d["filename"].astype(str)
    d["text"] = d["text"].astype(str).fillna("")
    return d[["filename","text"]]

subs = [load_sub(p) for p in SUB_PATHS]
base = subs[0].rename(columns={"text":"t0"})
for i, d in enumerate(subs[1:], start=1):
    base = base.merge(d.rename(columns={"text":f"t{i}"}), on="filename", how="inner")

cols = [c for c in base.columns if c.startswith("t")]
N = len(base)
print("✅ merged rows:", N, "| files:", len(cols))

# ===================== BUILD TRAIN LM (uni+bi) =====================
df = pd.read_csv(train_csv)
df = df[df["text"].notna()]
texts = df["text"].astype(str).tolist()

uni = Counter()
bi  = Counter()
total_uni = 0

for t in texts:
    toks = ["<s>"] + word_tokenize(t) + ["</s>"]
    if len(toks) < 2:
        continue
    for w in toks[1:]:
        uni[w] += 1
        total_uni += 1
    for a,b in zip(toks[:-1], toks[1:]):
        bi[(a,b)] += 1

V = max(1, len(uni))
print("✅ train lines:", len(texts))
print("✅ vocab words:", V, "| bigram types:", len(bi))

def lm_score(text: str) -> float:
    toks = ["<s>"] + word_tokenize(text) + ["</s>"]
    if len(toks) < 2:
        return -1e9
    k = 0.1
    s = 0.0
    for a,b in zip(toks[:-1], toks[1:]):
        c_ab = bi.get((a,b), 0)
        c_a  = uni.get(a, 0)
        prob = (c_ab + k) / (c_a + k*V + 1e-9)
        s += math.log(prob + 1e-12)
    # length norm (لا يفضل القصير جدًا)
    s /= max(1, (len(toks)-1))**0.70
    return s

# ===================== LAST SHOT CHOOSER =====================
MAJORITY_K = 3          # 3 من 5 = قرار حاسم
CONS_BONUS = 0.65       # bonus للاتفاق الجزئي (2/5..الخ)
SHORT_PEN  = 3.0        # عقوبة للفاضي/قصير
ALPHA_LM   = 1.0        # وزن LM (خليه 1.0 آخر محاولة)

picked_counts = Counter()
maj_hits = 0
chosen = []

for idx, row in enumerate(base.itertuples(index=False), start=1):
    cands = [getattr(row, c) for c in cols]
    cands = [light_post(x) for x in cands]

    normed = [norm_for_match(x) for x in cands]
    freq = Counter(normed)

    # 1) hard majority
    best_norm, best_freq = freq.most_common(1)[0]
    if best_freq >= MAJORITY_K and best_norm.strip():
        # اختر أول واحد مطابق (مهم عشان نحافظ على الشكل/الأرقام كما هي في ذلك المرشح)
        pick_i = next(i for i,nm in enumerate(normed) if nm == best_norm)
        chosen_text = cands[pick_i]
        picked_counts[pick_i] += 1
        maj_hits += 1
    else:
        # 2) LM + consensus bonus
        best_i = 0
        best_s = -1e18
        for i, t in enumerate(cands):
            s = ALPHA_LM * lm_score(t)
            s += CONS_BONUS * (freq[normed[i]] - 1)
            if len(t.strip()) < 3:
                s -= SHORT_PEN
            if s > best_s:
                best_s = s
                best_i = i
        chosen_text = cands[best_i]
        picked_counts[best_i] += 1

    chosen.append(chosen_text)

    if idx <= 3 or idx % 300 == 0:
        print("—"*95)
        print(f"row {idx}/{N} | file: {row.filename}")
        for k,t in enumerate(cands):
            print(f"[{k}] {t[:140]}")
        print("✅ chosen:", chosen_text[:160])
        print("freq max:", best_freq, "| maj_hits so far:", maj_hits)

print("\n✅ Majority hits:", maj_hits, "out of", N, f"({maj_hits/N*100:.1f}%)")
print("✅ Pick distribution:")
for k in range(len(cols)):
    print(f"file[{k}] -> {picked_counts[k]}")

# ===================== SAVE =====================
out_path = "/kaggle/working/submission_lastshot.csv"
out_df = pd.DataFrame({"filename": base["filename"], "text": chosen})
out_df.to_csv(out_path, index=False, encoding="utf-8-sig")
print("\n✅ saved:", out_path)
print(out_df.head(10))

✅ Using submissions:
1 /kaggle/input/datasets/geleefinal/top-reading/submission (8).csv
2 /kaggle/input/datasets/geleefinal/best-2/Final_Submission_Rank1.csv
3 /kaggle/input/datasets/geleefinal/best-3/Final_Submission_Gold_V5.csv
4 /kaggle/input/datasets/geleefinal/best-4/LAST.csv
5 /kaggle/input/datasets/geleefinal/best-5/submission (10).csv
✅ merged rows: 2671 | files: 5
✅ train lines: 15962
✅ vocab words: 41420 | bigram types: 131616
———————————————————————————————————————————————————————————————————————————————————————————————
row 1/2671 | file: 1958_p112_l0033.png
[0] إنهم لا يوافقون على قوة دولية ولا يستبدلون بالقوات الانكليزي المراقبين دولين
[1] انهم لا يوافتون علي قبوه دوليه ولا يستبدلون بالقوات الانكليزي المراقبين دولين
[2] نهم لا يوافقون عل قبو دولي ولا يستبدلون بالقوات الانكليزي المراقبين دولين
[3] إنهم لا يوافقون على قوة دولية ولا يستبدلون بالقوات الانكليزي المراقبين دولين
[4] إنهم لا يواخقون على قبوة دولية ولا يستبدلون بالقوات الانكليزي المراقبين دولين
✅ chosen: إنهم لا يو

In [27]:
import re, math
import pandas as pd
from collections import Counter

# ============ PATHS ============
A_PATH = "/kaggle/input/datasets/geleefinal/top-reading/submission (8).csv"   # الأفضل (0.1234)
B_PATH = "/kaggle/working/submission_ultra_pp.csv"                            # الثاني الأفضل عندك
TRAIN_CSV = "/kaggle/input/datasets/geleefinal/gelee-nakba/AR-MS.train/annotations.csv"

OUT_PATH = "/kaggle/working/submission_last_attack.csv"

print("✅ A:", A_PATH)
print("✅ B:", B_PATH)
print("✅ Train:", TRAIN_CSV)

# ============ NORMALIZE / TOKENIZE ============
AR_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
TATWEEL = "\u0640"

def norm_for_lm(s: str) -> str:
    s = str(s).replace(TATWEEL, "")
    s = AR_DIACRITICS.sub("", s)
    s = re.sub(r"\s+", " ", s).strip()
    # خفيف جدًا (بدون تخريب الأرقام/الهمزات بقوة)
    return s

def tokenize_words(s: str):
    s = norm_for_lm(s)
    s = re.sub(r"[\"“”‘’\(\)\[\]\{\}]", " ", s)
    s = re.sub(r"[،؛:!\?\.]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s.split() if s else []

def light_post(s: str) -> str:
    s = str(s)
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"\s+([،\.!\?؛:])", r"\1", s)
    return s

# ============ LOAD FILES ============
def load_sub(path: str) -> pd.DataFrame:
    d = pd.read_csv(path)
    if "filename" not in d.columns and "image" in d.columns:
        d = d.rename(columns={"image":"filename"})
    assert "filename" in d.columns and "text" in d.columns, f"Bad columns in {path}: {d.columns}"
    d["filename"] = d["filename"].astype(str)
    d["text"] = d["text"].astype(str).fillna("")
    return d[["filename","text"]]

A = load_sub(A_PATH).rename(columns={"text":"A"})
B = load_sub(B_PATH).rename(columns={"text":"B"})
df = A.merge(B, on="filename", how="inner")
print("✅ merged:", df.shape)

# ============ BUILD TRAIN LM (uni+bi) ============
train = pd.read_csv(TRAIN_CSV)
train = train[train["text"].notna()]
texts = train["text"].astype(str).tolist()

uni = Counter()
bi = Counter()
total_uni = 0

for t in texts:
    toks = ["<s>"] + tokenize_words(t) + ["</s>"]
    if len(toks) < 2: 
        continue
    for w in toks[1:]:
        uni[w] += 1
        total_uni += 1
    for a,bw in zip(toks[:-1], toks[1:]):
        bi[(a,bw)] += 1

V = max(1, len(uni))
print("✅ train vocab:", V, "| bigrams:", len(bi))

def lm_score(text: str) -> float:
    toks = ["<s>"] + tokenize_words(text) + ["</s>"]
    if len(toks) < 2:
        return -1e9
    k = 0.1
    s = 0.0
    for a,bw in zip(toks[:-1], toks[1:]):
        c_ab = bi.get((a,bw), 0)
        c_a  = uni.get(a, 0)
        p = (c_ab + k) / (c_a + k*V + 1e-9)
        s += math.log(p + 1e-12)
    # length norm (لا يفضل القصير)
    s /= max(1, (len(toks)-1))**0.70
    return s

# ============ WORD-COVERAGE SCORE (from train vocab) ============
def coverage_score(text: str) -> float:
    toks = tokenize_words(text)
    if not toks: 
        return -5.0
    hit = sum(1 for w in toks if w in uni)
    return hit / max(1, len(toks))

# ============ FINAL PICKER ============
# A default، لا نختار B إلا إذا كان B أحسن بفارق "Margin" واضح
LM_W   = 1.0
COV_W  = 2.0
LEN_W  = 0.02
MARGIN = 0.18     # كل ما زاد = أكثر محافظة (أقل مخاطرة)

picked_A = 0
picked_B = 0

out_texts = []
for i, r in enumerate(df.itertuples(index=False), start=1):
    a = light_post(r.A)
    b = light_post(r.B)

    sa = LM_W*lm_score(a) + COV_W*coverage_score(a) + LEN_W*len(a)
    sb = LM_W*lm_score(b) + COV_W*coverage_score(b) + LEN_W*len(b)

    # اختيار محافظ: B لازم يتفوق بفارق
    if sb > sa + MARGIN:
        out = b
        picked_B += 1
    else:
        out = a
        picked_A += 1

    out_texts.append(out)

    if i <= 3 or i % 500 == 0:
        print("—"*90)
        print(f"{i}/{len(df)} | {r.filename}")
        print("A:", a[:140])
        print("B:", b[:140])
        print(f"scoreA={sa:.3f} scoreB={sb:.3f} -> chosen:", ("B" if out==b else "A"))

print("\n✅ Picked A:", picked_A, "| Picked B:", picked_B)

out_df = pd.DataFrame({"filename": df["filename"], "text": out_texts})
out_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print("✅ saved:", OUT_PATH)
out_df.head(10)

✅ A: /kaggle/input/datasets/geleefinal/top-reading/submission (8).csv
✅ B: /kaggle/working/submission_ultra_pp.csv
✅ Train: /kaggle/input/datasets/geleefinal/gelee-nakba/AR-MS.train/annotations.csv
✅ merged: (2671, 3)
✅ train vocab: 42706 | bigrams: 132983
——————————————————————————————————————————————————————————————————————————————————————————
1/2671 | 1958_p112_l0033.png
A: إنهم لا يوافقون على قوة دولية ولا يستبدلون بالقوات الانكليزي المراقبين دولين
B: إنهم لا يوافقون على قوة دولية ولا يستبدلون بالقوات الانكليزي المراقبين دولين
scoreA=-18.768 scoreB=-18.768 -> chosen: B
——————————————————————————————————————————————————————————————————————————————————————————
2/2671 | 1964_p167_l0044.png
A: بوجد في السعودي ٧٠ معلمة أردنية تعلم في مدارس الأردن هذه السنة غير المرفوضات
B: بوجد في السعودي ٧٠ معلمة أردنية تعلم في مدارس الأردن هذه السنة غير المرفوضات
scoreA=-19.331 scoreB=-19.331 -> chosen: B
——————————————————————————————————————————————————————————————————————————————————————————
3/2671

,filename,text
0,1958_p112_l0033.png,إنهم لا يوافقون على قوة دولية ولا يستبدلون بال...
1,1964_p167_l0044.png,بوجد في السعودي ٧٠ معلمة أردنية تعلم في مدارس ...
2,1956_p080_l0012.png,النواب استقالت وإذا ل نالتمها ترعت في الحكم وب...
3,1960_p116_l0063.png,جستدقا بعرة فسيفته من مل تاحية وصوب ويشعراز بل...
4,1962_p077_l0032.png,د احي سشوايدو المرعة الذبي شمو بر عدير برمشعل ...
5,1964b_p077_l0044.png,بوجد في السعودي ٧٠ معلمة أردنية تعلم في مدارس ...
6,1958_p012_l0029.png,وعنده ذهب تقد رقيمته بمبلغي٠.٠ دنيار خير الجني...
7,1962_p173_l0055.png,(٣) الشاء محكمة عدل عليا اطلسية. وأقر المؤتمر ...
8,1954_p068_l0020.png,إن أمام هته العاصفة أختالاتا ١(١) أما آن تهدا ...
9,1960_p104_l0015.png,أة أمريكا تستورد ٦٠ب% من تاج كركوبا أو ماميادل...


In [28]:
import re, math
import pandas as pd
from collections import Counter

# ============ PATHS ============
A_PATH = "/kaggle/input/datasets/geleefinal/top-reading/submission (8).csv"
B_PATH = "/kaggle/working/submission_ultra_pp.csv"
TRAIN_CSV = "/kaggle/input/datasets/geleefinal/gelee-nakba/AR-MS.train/annotations.csv"

OUT_PATH = "/kaggle/working/submission_last_attack_ultra.csv"

print("🔥 ULTRA AGGRESSIVE MODE")
print("A:", A_PATH)
print("B:", B_PATH)

# ============ NORMALIZATION ============
AR_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
TATWEEL = "\u0640"

def norm_for_lm(s: str) -> str:
    s = str(s).replace(TATWEEL, "")
    s = AR_DIACRITICS.sub("", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize_words(s: str):
    s = norm_for_lm(s)
    s = re.sub(r"[\"“”‘’\(\)\[\]\{\}]", " ", s)
    s = re.sub(r"[،؛:!\?\.]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s.split() if s else []

def light_post(s: str) -> str:
    s = str(s)
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"\s+([،\.!\?؛:])", r"\1", s)
    return s

# ============ LOAD ============
def load_sub(path: str) -> pd.DataFrame:
    d = pd.read_csv(path)
    if "filename" not in d.columns and "image" in d.columns:
        d = d.rename(columns={"image":"filename"})
    d["filename"] = d["filename"].astype(str)
    d["text"] = d["text"].astype(str).fillna("")
    return d[["filename","text"]]

A = load_sub(A_PATH).rename(columns={"text":"A"})
B = load_sub(B_PATH).rename(columns={"text":"B"})
df = A.merge(B, on="filename", how="inner")

print("Merged rows:", df.shape[0])

# ============ BUILD TRAIN LM ============
train = pd.read_csv(TRAIN_CSV)
train = train[train["text"].notna()]
texts = train["text"].astype(str).tolist()

uni = Counter()
bi = Counter()

for t in texts:
    toks = ["<s>"] + tokenize_words(t) + ["</s>"]
    for w in toks[1:]:
        uni[w] += 1
    for a,bw in zip(toks[:-1], toks[1:]):
        bi[(a,bw)] += 1

V = max(1, len(uni))
print("Train vocab:", V)

def lm_score(text: str) -> float:
    toks = ["<s>"] + tokenize_words(text) + ["</s>"]
    if len(toks) < 2:
        return -1e9
    k = 0.1
    s = 0.0
    for a,bw in zip(toks[:-1], toks[1:]):
        c_ab = bi.get((a,bw), 0)
        c_a  = uni.get(a, 0)
        p = (c_ab + k) / (c_a + k*V + 1e-9)
        s += math.log(p + 1e-12)
    s /= max(1, (len(toks)-1))**0.65
    return s

def coverage_score(text: str) -> float:
    toks = tokenize_words(text)
    if not toks:
        return -5.0
    hit = sum(1 for w in toks if w in uni)
    return hit / max(1, len(toks))

# ============ ULTRA SETTINGS ============
LM_W   = 1.2
COV_W  = 3.0
LEN_W  = 0.01
MARGIN = -0.05   # 🔥 هجومي جدًا (B يختار بسهولة)

picked_A = 0
picked_B = 0
out_texts = []

for i, r in enumerate(df.itertuples(index=False), start=1):
    a = light_post(r.A)
    b = light_post(r.B)

    sa = LM_W*lm_score(a) + COV_W*coverage_score(a) + LEN_W*len(a)
    sb = LM_W*lm_score(b) + COV_W*coverage_score(b) + LEN_W*len(b)

    if sb > sa + MARGIN:
        out = b
        picked_B += 1
    else:
        out = a
        picked_A += 1

    out_texts.append(out)

    if i <= 3 or i % 500 == 0:
        print("—"*80)
        print(f"{i}/{len(df)} | {r.filename}")
        print("A:", a[:120])
        print("B:", b[:120])
        print(f"scoreA={sa:.3f} scoreB={sb:.3f} -> chosen:", "B" if out==b else "A")

print("\nPicked A:", picked_A)
print("Picked B:", picked_B)

out_df = pd.DataFrame({"filename": df["filename"], "text": out_texts})
out_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print("\n🔥 ULTRA FILE SAVED:", OUT_PATH)

🔥 ULTRA AGGRESSIVE MODE
A: /kaggle/input/datasets/geleefinal/top-reading/submission (8).csv
B: /kaggle/working/submission_ultra_pp.csv
Merged rows: 2671
Train vocab: 42706
————————————————————————————————————————————————————————————————————————————————
1/2671 | 1958_p112_l0033.png
A: إنهم لا يوافقون على قوة دولية ولا يستبدلون بالقوات الانكليزي المراقبين دولين
B: إنهم لا يوافقون على قوة دولية ولا يستبدلون بالقوات الانكليزي المراقبين دولين
scoreA=-26.713 scoreB=-26.713 -> chosen: B
————————————————————————————————————————————————————————————————————————————————
2/2671 | 1964_p167_l0044.png
A: بوجد في السعودي ٧٠ معلمة أردنية تعلم في مدارس الأردن هذه السنة غير المرفوضات
B: بوجد في السعودي ٧٠ معلمة أردنية تعلم في مدارس الأردن هذه السنة غير المرفوضات
scoreA=-27.674 scoreB=-27.674 -> chosen: B
————————————————————————————————————————————————————————————————————————————————
3/2671 | 1956_p080_l0012.png
A: النواب استقالت وإذا ل نالتمها ترعت في الحكم وبأرشرت اعمالها في شرم ورضاتة أما أن الملك يط

In [29]:
import pandas as pd
import re

BEST_PATH = "/kaggle/working/submission_lastshot.csv"  # الأفضل 0.1138
P2_PATH   = "/kaggle/working/submission_last_attack_ultra.csv"
P3_PATH   = "/kaggle/input/datasets/geleefinal/top-reading/submission (8).csv"

def load_sub(path):
    df = pd.read_csv(path)
    if "filename" not in df.columns and "image" in df.columns:
        df = df.rename(columns={"image":"filename"})
    df["filename"] = df["filename"].astype(str)
    df["text"] = df["text"].astype(str).fillna("")
    return df[["filename","text"]].sort_values("filename").reset_index(drop=True)

best = load_sub(BEST_PATH).rename(columns={"text":"t_best"})
p2   = load_sub(P2_PATH).rename(columns={"text":"t_p2"})
p3   = load_sub(P3_PATH).rename(columns={"text":"t_p3"})

df = best.merge(p2, on="filename", how="inner").merge(p3, on="filename", how="inner")
print("✅ rows:", len(df))
print("✅ columns:", df.columns.tolist())
print("\n🔎 quick peek:")
print(df.head(3))

✅ rows: 2671
✅ columns: ['filename', 't_best', 't_p2', 't_p3']

🔎 quick peek:
               filename                                             t_best  \
0  1953a_p007_l0024.png  بعثة أخرى من فوق راسي وتخطوا الشرف والتقاليد ا...   
1  1953a_p018_l0010.png  ثم انتقلنا الي النادي وقد انصرف كثير من رواده ...   
2  1953a_p018_l0020.png  أيدي افراد أو جماعات (شركات )بل يجب أن تسيطر ع...   

                                                t_p2  \
0  بصثة أخرى منفوق رأسي وتخطوا الشرف والتقاليد ال...   
1  ثم انتقلنا إلى النادي وقد انصرف كثير من رواده ...   
2  أيدي افراد أو جماعات (شركات )بل يجب أن تسيطر ع...   

                                                t_p3  
0  بصثة أخرى منفوق رأسي وتخطوا الشرف والتقاليد ال...  
1  ثم انتقلنا إلى النادي وقد انصرف كثير من رواده ...  
2  أيدي افراد أو جماعات (شركات )بل يجب أن تسيطر ع...  


In [30]:
import math
from collections import Counter
from tqdm import tqdm

# أوزان (نثق بالأفضل أكثر)
W_BEST = 1.00
W_P2   = 0.65
W_P3   = 0.70

def light_post(s: str) -> str:
    # تنظيف آمن جدًا (ما يغيّر حروف بشكل هجومي)
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"\s+([،\.!\?؛:])", r"\1", s)
    s = re.sub(r"([،\.!\?؛:])(?=\S)", r"\1 ", s)  # مسافة بعد الترقيم إذا بعدها حرف
    s = re.sub(r"\s+", " ", s).strip()
    return s

def score_text(t: str) -> float:
    # سكور بسيط لتفضيل النص “المعقول” (بدون تخريب)
    t = t.strip()
    if not t:
        return -1e9
    # عقوبات خفيفة
    bad_spaces = len(re.findall(r"\s{2,}", t))
    weird_rep  = len(re.findall(r"(.)\1{3,}", t))
    # تفضيل طول متوسط
    L = len(t)
    return (
        0.015 * min(L, 220)     # طول مفيد
        - 0.30 * bad_spaces
        - 0.40 * weird_rep
    )

def pick_ensemble(row, margin: float):
    # تجميع مرشحين (قد تتكرر بين الملفات)
    cands = []
    t1 = light_post(row.t_best)
    t2 = light_post(row.t_p2)
    t3 = light_post(row.t_p3)

    cands.append((t1, W_BEST))
    cands.append((t2, W_P2))
    cands.append((t3, W_P3))

    # نجمع أوزان النصوص المتطابقة
    agg = {}
    for t, w in cands:
        agg[t] = agg.get(t, 0.0) + w

    # نحسب سكور نهائي = وزن تصويت + سكور لغوي بسيط
    scored = []
    for t, wsum in agg.items():
        scored.append((t, wsum + score_text(t)))

    scored.sort(key=lambda x: x[1], reverse=True)
    best_t, best_s = scored[0]
    if len(scored) == 1:
        return best_t, scored

    second_t, second_s = scored[1]

    # margin gating: إذا الفرق بسيط نرجع للأفضل الأساسي (عشان نقلل مخاطرة)
    if (best_s - second_s) < margin:
        return t1, scored  # ارجعي لملفك الأفضل (0.1138)
    return best_t, scored

# مستويات الهجوم:
# low: محافظ جدًا (يرجع للأفضل إذا فيه شك)
# mid: متوسط
# high: هجومي أكثر
MARGINS = {
    "low": 0.55,
    "mid": 0.30,
    "high": 0.10,
}

outs = {k: [] for k in MARGINS.keys()}

print("🚀 Building 3 submissions (Low/Mid/High) ...")
for i, row in enumerate(tqdm(df.itertuples(index=False), total=len(df), desc="Ensembling")):
    for k, m in MARGINS.items():
        chosen, scored = pick_ensemble(row, margin=m)
        outs[k].append((row.filename, chosen))

    # طباعة أول 3 أمثلة فقط
    if i < 3:
        print("\n" + "—"*90)
        print("File:", row.filename)
        print("BEST:", light_post(row.t_best)[:160])
        print("P2  :", light_post(row.t_p2)[:160])
        print("P3  :", light_post(row.t_p3)[:160])
        for k, m in MARGINS.items():
            chosen, scored = pick_ensemble(row, margin=m)
            print(f"✅ {k.upper()} (margin={m}): {chosen[:160]}")

# احفظ الملفات
for k in outs:
    out_df = pd.DataFrame(outs[k], columns=["filename","text"])
    out_path = f"/kaggle/working/submission_ens_{k}.csv"
    out_df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print("✅ saved:", out_path, "| rows:", len(out_df))

🚀 Building 3 submissions (Low/Mid/High) ...


Ensembling:  21%|██        | 551/2671 [00:00<00:00, 5508.19it/s]


——————————————————————————————————————————————————————————————————————————————————————————
File: 1953a_p007_l0024.png
BEST: بعثة أخرى من فوق راسي وتخطوا الشرف والتقاليد الدولية والوفاء وهؤلاء كم قد موالي
P2  : بصثة أخرى منفوق رأسي وتخطوا الشرف والتقاليد الدولية والوفاء وهؤلاء كم قدموا لي
P3  : بصثة أخرى منفوق رأسي وتخطوا الشرف والتقاليد الدولية والوفاء وهؤلاء كم قدموا لي
✅ LOW (margin=0.55): بعثة أخرى من فوق راسي وتخطوا الشرف والتقاليد الدولية والوفاء وهؤلاء كم قد موالي
✅ MID (margin=0.3): بصثة أخرى منفوق رأسي وتخطوا الشرف والتقاليد الدولية والوفاء وهؤلاء كم قدموا لي
✅ HIGH (margin=0.1): بصثة أخرى منفوق رأسي وتخطوا الشرف والتقاليد الدولية والوفاء وهؤلاء كم قدموا لي

——————————————————————————————————————————————————————————————————————————————————————————
File: 1953a_p018_l0010.png
BEST: ثم انتقلنا الي النادي وقد انصرف كثير من رواده لان شجارا نشب بين الدكتور
P2  : ثم انتقلنا إلى النادي وقد انصرف كثير من رواده لأن شجارا لشعب بين الدكتور
P3  : ثم انتقلنا إلى النادي وقد انصرف كثير من روا

Ensembling: 100%|██████████| 2671/2671 [00:00<00:00, 6476.85it/s]

✅ saved: /kaggle/working/submission_ens_low.csv | rows: 2671
✅ saved: /kaggle/working/submission_ens_mid.csv | rows: 2671
✅ saved: /kaggle/working/submission_ens_high.csv | rows: 2671


In [31]:
def compare_to_best(best_path, new_path, show=12):
    best = load_sub(best_path).rename(columns={"text":"best"})
    new  = load_sub(new_path).rename(columns={"text":"new"})
    m = best.merge(new, on="filename", how="inner")
    changed = (m["best"] != m["new"]).sum()
    total = len(m)
    print(f"\n📌 Compare:\nBEST: {best_path}\nNEW : {new_path}")
    print(f"✅ Changed lines: {changed}/{total} = {changed/total*100:.2f}%")

    diffs = m[m["best"] != m["new"]].head(show)
    if len(diffs) == 0:
        print("✅ No differences.")
        return
    print(f"\n🔎 first {min(show,len(diffs))} diffs:")
    for r in diffs.itertuples(index=False):
        print("—"*90)
        print(r.filename)
        print("BEST:", r.best[:170])
        print("NEW :", r.new[:170])

# قارني الثلاثة
compare_to_best(BEST_PATH, "/kaggle/working/submission_ens_low.csv")
compare_to_best(BEST_PATH, "/kaggle/working/submission_ens_mid.csv")
compare_to_best(BEST_PATH, "/kaggle/working/submission_ens_high.csv")


📌 Compare:
BEST: /kaggle/working/submission_lastshot.csv
NEW : /kaggle/working/submission_ens_low.csv
✅ Changed lines: 21/2671 = 0.79%

🔎 first 12 diffs:
——————————————————————————————————————————————————————————————————————————————————————————
1955_p126_l0031.png
BEST: بين العرب واليهود وحرض فيه ثلاث نقط بارزة ليست جديرة بأي بحث أو قبول:.
NEW : بين العرب واليهود وحرض فيه ثلاث نقط بارزة ليست جديرة بأي بحث أو قبول: .
——————————————————————————————————————————————————————————————————————————————————————————
1956_p041_l0006.png
BEST: محطة ب" الجفور" وقد تحريت أسبناب هذه النقمة فقيلني:نهم بتشيوعيون وقيل أنها احركواتع
NEW : محطة ب" الجفور" وقد تحريت أسبناب هذه النقمة فقيلني: نهم بتشيوعيون وقيل أنها احركواتع
——————————————————————————————————————————————————————————————————————————————————————————
1956_p071_l0056.png
BEST: الجلاء:يسي قدم الاليز الظالمون واتلوا مصر وعدوا أن يركوحاعالما يستب الأمن وكروا
NEW : الجلاء: يسي قدم الاليز الظالمون واتلوا مصر وعدوا أن يركوحاعالما يستب الأمن وكروا
———

In [32]:
import pandas as pd
import re
from collections import Counter

TRAIN_CSV = "/kaggle/input/datasets/geleefinal/gelee-nakba/AR-MS.train/annotations.csv"
SUB_PATH  = "/kaggle/working/submission_lastshot.csv"

def tokenize(text):
    return re.findall(r"[اأإآء-ي0-9٠-٩]+", str(text))

# تحميل التدريب
train = pd.read_csv(TRAIN_CSV)
train_words = []
for t in train["text"].dropna():
    train_words.extend(tokenize(t))

train_vocab = set(train_words)

# تحميل السبمشن
sub = pd.read_csv(SUB_PATH)
sub_words = []
for t in sub["text"].dropna():
    sub_words.extend(tokenize(t))

sub_vocab = set(sub_words)

print("📊 TRAIN")
print("Total words:", len(train_words))
print("Unique words:", len(train_vocab))

print("\n📊 SUBMISSION")
print("Total words:", len(sub_words))
print("Unique words:", len(sub_vocab))

# كلمات غير موجودة في التدريب
oov = sub_vocab - train_vocab
print("\n📌 Words in submission NOT in train:", len(oov))
print("OOV percentage:", round(len(oov)/len(sub_vocab)*100,2), "%")

print("\n🔎 Sample OOV words:")
print(list(oov)[:20])

📊 TRAIN
Total words: 159749
Unique words: 42270

📊 SUBMISSION
Total words: 29587
Unique words: 15352

📌 Words in submission NOT in train: 8316
OOV percentage: 54.17 %

🔎 Sample OOV words:
['ويطمويوما', 'جربحة', 'انفسخا', 'فذوى', 'دحمزة', 'وفقتل', 'وغطت', 'ويخضوا', 'جامق', 'لتيدي', 'بكبنت٧٠', 'الشجاب', 'المتلة', 'مصيدق', 'يتعضب', 'ينيتهي', 'بمعانيها', 'الخارجيه', 'وتصهرهم', 'معايدات']


In [33]:
import pandas as pd, re, os
from collections import Counter, defaultdict
from tqdm import tqdm

# ========= paths (عدّلي فقط هذي) =========
TRAIN_CSV = "/kaggle/input/datasets/geleefinal/gelee-nakba/AR-MS.train/annotations.csv"
IN_SUB    = "/kaggle/working/submission_lastshot.csv"   # أفضل ملفك (0.1138)
OUT_SUB   = "/kaggle/working/submission_dotfix_attack.csv"

print("✅ TRAIN_CSV:", TRAIN_CSV)
print("✅ IN_SUB   :", IN_SUB)

# ========= tokenization =========
WORD_RE = re.compile(r"[اأإآء-ي0-9٠-٩]+")  # كلمات عربية/أرقام

def tokenize_with_spans(text):
    # يرجع [(token, start, end), ...]
    return [(m.group(0), m.start(), m.end()) for m in WORD_RE.finditer(str(text))]

# ========= undot mapping (رسم بدون نقاط) =========
DOT_BASE = {
    # ب/ت/ث
    "ب":"ب","ت":"ب","ث":"ب",
    # ج/ح/خ
    "ج":"ح","ح":"ح","خ":"ح",
    # د/ذ
    "د":"د","ذ":"د",
    # ر/ز
    "ر":"ر","ز":"ر",
    # س/ش
    "س":"س","ش":"س",
    # ص/ض
    "ص":"ص","ض":"ص",
    # ط/ظ
    "ط":"ط","ظ":"ط",
    # ع/غ
    "ع":"ع","غ":"ع",
    # ف/ق
    "ف":"ف","ق":"ف",
    # ن/ي/ى/ئ (قريبة جدًا في مخطوطات)
    "ن":"ن","ي":"ن","ى":"ن","ئ":"ن",
    # ه/ة
    "ه":"ه","ة":"ه",
}

def undot(word: str) -> str:
    return "".join(DOT_BASE.get(ch, ch) for ch in word)

# ========= conservative "is this only dot-confusions?" =========
DOT_FAMILIES = [
    set("بتث"), set("جحخ"), set("دذ"), set("رز"), set("سش"), set("صض"),
    set("طظ"), set("عغ"), set("فق"), set("نيىئ"), set("هة"),
]

def is_dot_family_change(a, b):
    # يضمن إن كل اختلافات الحروف داخل "عائلات" مسموحة
    if len(a) != len(b):
        return False
    diffs = [(ca, cb) for ca, cb in zip(a, b) if ca != cb]
    if not diffs:
        return False
    for ca, cb in diffs:
        ok = False
        for fam in DOT_FAMILIES:
            if ca in fam and cb in fam:
                ok = True
                break
        if not ok:
            return False
    return True

# ========= build train vocab + undot index =========
train = pd.read_csv(TRAIN_CSV)
train_texts = train["text"].dropna().astype(str).tolist()

train_words = []
for t in train_texts:
    train_words.extend([w for w,_,_ in tokenize_with_spans(t)])

train_freq = Counter(train_words)
train_vocab = set(train_freq.keys())

undot_to_cands = defaultdict(list)
for w, c in train_freq.items():
    u = undot(w)
    undot_to_cands[u].append((w, c))

# sort candidates by frequency desc
for u in list(undot_to_cands.keys()):
    undot_to_cands[u].sort(key=lambda x: x[1], reverse=True)

print("✅ train unique words:", len(train_vocab))
print("✅ undot keys:", len(undot_to_cands))

# ========= scoring / selection =========
def choose_replacement(word):
    # لا نلمس الكلمات الموجودة فعلاً في التدريب (لتقليل خراب WER)
    if word in train_vocab:
        return None

    u = undot(word)
    cands = undot_to_cands.get(u, None)
    if not cands:
        return None

    # ناخذ أعلى مرشحين (مثلاً top 8) ونفلتر
    best = None
    for cand, freq in cands[:8]:
        if len(cand) != len(word):
            continue
        # نفس الرسم بدون نقاط (مضمون هنا) + اختلاف نقاط فقط
        if not is_dot_family_change(word, cand):
            continue
        # لا نغير لو الاختلاف كبير (أكثر من حرفين)
        diff_count = sum(1 for a,b in zip(word, cand) if a != b)
        if diff_count > 2:
            continue
        # شرط إضافي: المرشح لازم يكون "شائع" شوي في التدريب
        if freq < 3:
            continue
        best = cand
        break

    return best

# ========= apply to submission =========
sub = pd.read_csv(IN_SUB)
assert "filename" in sub.columns and "text" in sub.columns

total_tokens = 0
changed_tokens = 0
examples = []

new_texts = []
for i, text in enumerate(tqdm(sub["text"].astype(str).tolist(), desc="Dot-fix attack")):
    spans = tokenize_with_spans(text)
    total_tokens += len(spans)

    if not spans:
        new_texts.append(text)
        continue

    chars = list(text)
    # نطبق من الآخر للأول عشان الـ spans ما تتلخبط
    local_changes = []
    for w, s, e in reversed(spans):
        rep = choose_replacement(w)
        if rep and rep != w:
            # replace slice
            chars[s:e] = list(rep)
            changed_tokens += 1
            local_changes.append((w, rep))
            if len(examples) < 25:
                examples.append((sub.loc[i, "filename"], w, rep, text))

    new_text = "".join(chars)
    new_texts.append(new_text)

    # طباعة تقدم كل 500 سطر
    if (i+1) % 500 == 0:
        print(f"\n🟦 progress: {i+1}/{len(sub)} | changed_tokens={changed_tokens} | total_tokens={total_tokens}")

sub_out = sub.copy()
sub_out["text"] = new_texts
sub_out.to_csv(OUT_SUB, index=False, encoding="utf-8-sig")

print("\n✅ DONE")
print("Total tokens:", total_tokens)
print("Changed tokens:", changed_tokens)
print("Change rate:", round(100*changed_tokens/max(total_tokens,1), 3), "%")
print("Saved:", OUT_SUB)

print("\n🔎 Examples (first 25 changes):")
for fn, oldw, neww, line in examples[:25]:
    print(f"- {fn}: {oldw} -> {neww}")

✅ TRAIN_CSV: /kaggle/input/datasets/geleefinal/gelee-nakba/AR-MS.train/annotations.csv
✅ IN_SUB   : /kaggle/working/submission_lastshot.csv
✅ train unique words: 42270
✅ undot keys: 39334


Dot-fix attack: 100%|██████████| 2671/2671 [00:00<00:00, 45778.79it/s]


🟦 progress: 500/2671 | changed_tokens=223 | total_tokens=6775

🟦 progress: 1000/2671 | changed_tokens=404 | total_tokens=13163

🟦 progress: 1500/2671 | changed_tokens=533 | total_tokens=18208

🟦 progress: 2000/2671 | changed_tokens=653 | total_tokens=23041

🟦 progress: 2500/2671 | changed_tokens=757 | total_tokens=27980

✅ DONE
Total tokens: 29587
Changed tokens: 798
Change rate: 2.697 %
Saved: /kaggle/working/submission_dotfix_attack.csv

🔎 Examples (first 25 changes):
- 1960_p116_l0063.png: ناخيه -> ناحية
- 1962_p077_l0032.png: شقبه -> شقبة
- 1958_p012_l0029.png: دنيار -> دينار
- 1962_p173_l0055.png: محكمه -> محكمة
- 1954_p068_l0020.png: تهدا -> بهذا
- 1954_p068_l0020.png: العاصفه -> العاصفة
- 1960_p104_l0015.png: خن -> حي
- 1955_p174_l0031.png: سون -> سوى
- 1954_p178_l0021.png: الأفراج -> الأفراح
- 1958_p143_l0035.png: الفومي -> القومي
- 1958_p143_l0035.png: بدب -> تدب
- 1953b_p001_l0015.png: قرت -> قرب
- 1964_p113_l0046.png: غز -> عز
- 1964_p193_l0050.png: شاريا -> ساريا
- 1964b_p

In [34]:
import pandas as pd, re
from collections import Counter, defaultdict
from tqdm import tqdm

TRAIN_CSV = "/kaggle/input/datasets/geleefinal/gelee-nakba/AR-MS.train/annotations.csv"
IN_SUB    = "/kaggle/working/submission_lastshot.csv"
OUT_SUB   = "/kaggle/working/submission_dotfix_mid_smart.csv"

WORD_RE = re.compile(r"[اأإآء-ي0-9٠-٩]+")

def tokenize_with_spans(text):
    return [(m.group(0), m.start(), m.end()) for m in WORD_RE.finditer(str(text))]

DOT_BASE = {
    "ب":"ب","ت":"ب","ث":"ب",
    "ج":"ح","ح":"ح","خ":"ح",
    "د":"د","ذ":"د",
    "ر":"ر","ز":"ر",
    "س":"س","ش":"س",
    "ص":"ص","ض":"ص",
    "ط":"ط","ظ":"ط",
    "ع":"ع","غ":"ع",
    "ف":"ف","ق":"ف",
    "ن":"ن","ي":"ن","ى":"ن","ئ":"ن",
    "ه":"ه","ة":"ه",
}

def undot(word):
    return "".join(DOT_BASE.get(ch, ch) for ch in word)

DOT_FAMILIES = [
    set("بتث"), set("جحخ"), set("دذ"), set("رز"), set("سش"),
    set("صض"), set("طظ"), set("عغ"), set("فق"),
    set("نيىئ"), set("هة"),
]

def is_dot_family_change(a, b):
    if len(a) != len(b): return False
    diffs = [(ca, cb) for ca, cb in zip(a, b) if ca != cb]
    if not diffs: return False
    for ca, cb in diffs:
        ok = any(ca in fam and cb in fam for fam in DOT_FAMILIES)
        if not ok: return False
    return True

# ===== build vocab =====
train = pd.read_csv(TRAIN_CSV)
train_words = []
for t in train["text"].dropna().astype(str):
    train_words.extend([w for w,_,_ in tokenize_with_spans(t)])

train_freq = Counter(train_words)
train_vocab = set(train_freq.keys())

undot_to_cands = defaultdict(list)
for w, c in train_freq.items():
    undot_to_cands[undot(w)].append((w, c))

for u in undot_to_cands:
    undot_to_cands[u].sort(key=lambda x: x[1], reverse=True)

# ===== replacement logic =====
def choose_replacement(word):
    if word in train_vocab:
        return None

    cands = undot_to_cands.get(undot(word))
    if not cands:
        return None

    for cand, freq in cands[:12]:
        if len(cand) != len(word):
            continue
        if not is_dot_family_change(word, cand):
            continue

        diff_count = sum(1 for a,b in zip(word, cand) if a != b)

        if diff_count > 3:
            continue

        # حماية WER: لا نسمح بـ 3 تغييرات إلا لو الكلمة طويلة
        if diff_count == 3 and len(word) < 7:
            continue

        if freq < 2:
            continue

        return cand
    return None

# ===== apply =====
sub = pd.read_csv(IN_SUB)

total_tokens = 0
changed_tokens = 0
examples = []

new_texts = []

for i, text in enumerate(tqdm(sub["text"].astype(str), desc="MID Smart")):
    spans = tokenize_with_spans(text)
    total_tokens += len(spans)

    chars = list(text)

    for w, s, e in reversed(spans):
        rep = choose_replacement(w)
        if rep and rep != w:
            chars[s:e] = list(rep)
            changed_tokens += 1
            if len(examples) < 20:
                examples.append((sub.loc[i, "filename"], w, rep))

    new_texts.append("".join(chars))

sub["text"] = new_texts
sub.to_csv(OUT_SUB, index=False, encoding="utf-8-sig")

print("\n✅ DONE")
print("Total tokens:", total_tokens)
print("Changed tokens:", changed_tokens)
print("Change rate:", round(100*changed_tokens/max(total_tokens,1),3), "%")
print("Saved:", OUT_SUB)

print("\n🔎 Sample changes:")
for fn, o, n in examples:
    print(f"- {fn}: {o} -> {n}")

MID Smart: 100%|██████████| 2671/2671 [00:00<00:00, 45774.68it/s]


✅ DONE
Total tokens: 29587
Changed tokens: 1013
Change rate: 3.424 %
Saved: /kaggle/working/submission_dotfix_mid_smart.csv

🔎 Sample changes:
- 1960_p116_l0063.png: ناخيه -> ناحية
- 1960_p116_l0063.png: يغته -> نعته
- 1962_p077_l0032.png: شقبه -> شقبة
- 1958_p012_l0029.png: دنيار -> دينار
- 1962_p173_l0055.png: محكمه -> محكمة
- 1954_p068_l0020.png: تهدا -> بهذا
- 1954_p068_l0020.png: العاصفه -> العاصفة
- 1960_p104_l0015.png: خن -> حي
- 1955_p174_l0031.png: سون -> سوى
- 1954_p178_l0021.png: الأفراج -> الأفراح
- 1958_p143_l0035.png: الفومي -> القومي
- 1958_p143_l0035.png: بدب -> تدب
- 1953b_p001_l0015.png: قرت -> قرب
- 1964_p113_l0046.png: غز -> عز
- 1964_p193_l0050.png: شاريا -> ساريا
- 1964_p193_l0050.png: لقمه -> لقمة
- 1964b_p098_l0035.png: سله -> سلة
- 1964b_p098_l0035.png: حاء -> جاء
- 1964_p189_l0029.png: عربيه -> عربية
- 1956_p089_l0024.png: بكلمه -> بكلمة


In [35]:
import pandas as pd, re
from collections import Counter
from tqdm import tqdm

# ===== paths =====
TRAIN_CSV = "/kaggle/input/datasets/geleefinal/gelee-nakba/AR-MS.train/annotations.csv"
IN_SUB    = "/kaggle/working/submission_lastshot.csv"
OUT_SUB   = "/kaggle/working/submission_split_attack.csv"

# ===== regex =====
WORD_RE = re.compile(r"[اأإآء-ي0-9٠-٩]+")  # كلمات/أرقام فقط (نترك الباقي كما هو)

# ===== undot mapping (اختياري يساعد لو الفرق بالنقط) =====
DOT_BASE = {
    "ب":"ب","ت":"ب","ث":"ب",
    "ج":"ح","ح":"ح","خ":"ح",
    "د":"د","ذ":"د",
    "ر":"ر","ز":"ر",
    "س":"س","ش":"س",
    "ص":"ص","ض":"ص",
    "ط":"ط","ظ":"ط",
    "ع":"ع","غ":"ع",
    "ف":"ف","ق":"ف",
    "ن":"ن","ي":"ن","ى":"ن","ئ":"ن",
    "ه":"ه","ة":"ه",
}
def undot(w): return "".join(DOT_BASE.get(ch, ch) for ch in w)

# ===== load train vocab + freq =====
train = pd.read_csv(TRAIN_CSV)
train_words = []
for t in train["text"].dropna().astype(str):
    train_words.extend([m.group(0) for m in WORD_RE.finditer(t)])

freq = Counter(train_words)
vocab = set(freq.keys())

# undotted sets (لتسهيل المطابقة مع اختلاف النقط)
vocab_undot = {}
for w,c in freq.items():
    u = undot(w)
    # نخزن أقوى كلمة (أكثر تكرارًا) لكل undot
    if u not in vocab_undot or c > vocab_undot[u][1]:
        vocab_undot[u] = (w,c)

def best_from_undot(u):
    x = vocab_undot.get(u)
    return x[0] if x else None

print("✅ train unique words:", len(vocab))
print("✅ train total words :", sum(freq.values()))

# ===== split chooser (آمن) =====
MIN_FREQ_PART = 3      # ارفعيها لـ 5 لو تبين أمان أكثر
MIN_LEN_TOKEN = 6      # لا نقسم القصير
MAX_SPLITS_TRY = 40    # حماية من بطء

def choose_split(token):
    # لا نمس الأرقام/القصير
    if len(token) < MIN_LEN_TOKEN:
        return None
    if token.isdigit():
        return None

    # إذا أصلاً كلمة موجودة لا نقسمها
    if token in vocab:
        return None

    # نجرب تقسيمات
    # نقاط التقسيم: من 2 إلى len-2
    tried = 0
    best = None
    best_score = -1

    for k in range(2, len(token)-1):
        tried += 1
        if tried > MAX_SPLITS_TRY:
            break

        a = token[:k]
        b = token[k:]

        if len(a) < 2 or len(b) < 2:
            continue

        # مطابقة مباشرة أو undot
        a_ok = (a in vocab)
        b_ok = (b in vocab)

        a2 = a
        b2 = b

        if not a_ok:
            aa = best_from_undot(undot(a))
            if aa: a2, a_ok = aa, True
        if not b_ok:
            bb = best_from_undot(undot(b))
            if bb: b2, b_ok = bb, True

        if not (a_ok and b_ok):
            continue

        fa = freq.get(a2, 0)
        fb = freq.get(b2, 0)

        # شرط أمان: لازم تكرار كافي
        if fa < MIN_FREQ_PART or fb < MIN_FREQ_PART:
            continue

        # Score: نجمع لوغاريتمي بسيط (بدون numpy)
        score = (fa + fb)

        # تفضيل تقسيم أقرب للنصف (يقلل تقسيمات غريبة)
        balance_penalty = abs(len(a) - len(b))
        score = score - 0.5 * balance_penalty

        if score > best_score:
            best_score = score
            best = (a2, b2)

    return best  # (word1, word2) أو None

# ===== apply on submission =====
sub = pd.read_csv(IN_SUB)

changed_tokens = 0
total_tokens = 0
examples = []

new_texts = []

for i, text in enumerate(tqdm(sub["text"].astype(str), desc="Split-attack (safe)")):
    spans = [(m.group(0), m.start(), m.end()) for m in WORD_RE.finditer(text)]
    total_tokens += len(spans)

    chars = list(text)

    # نعالج من النهاية للبداية عشان الإندكس ما يلخبط
    for tok, s, e in reversed(spans):
        split = choose_split(tok)
        if split:
            a2, b2 = split
            repl = f"{a2} {b2}"
            chars[s:e] = list(repl)
            changed_tokens += 1
            if len(examples) < 25:
                examples.append((sub.loc[i, "filename"], tok, repl))

    new_texts.append("".join(chars))

sub["text"] = new_texts
sub.to_csv(OUT_SUB, index=False, encoding="utf-8-sig")

print("\n✅ DONE split attack")
print("Total tokens:", total_tokens)
print("Changed tokens:", changed_tokens)
print("Change rate:", round(100*changed_tokens/max(total_tokens,1), 3), "%")
print("Saved:", OUT_SUB)

print("\n🔎 Sample splits:")
for fn, old, new in examples:
    print(f"- {fn}: {old} -> {new}")

✅ train unique words: 42270
✅ train total words : 159749


Split-attack (safe): 100%|██████████| 2671/2671 [00:00<00:00, 29200.91it/s]


✅ DONE split attack
Total tokens: 29587
Changed tokens: 656
Change rate: 2.217 %
Saved: /kaggle/working/submission_split_attack.csv

🔎 Sample splits:
- 1958_p112_l0033.png: بالقوات -> بال قوات
- 1964_p167_l0044.png: السعودينة -> السعود نية
- 1964_p167_l0044.png: بوجذفي -> توجد في
- 1956_p080_l0012.png: ورضاته -> ورضا به
- 1956_p080_l0012.png: اعمالها -> اعمال ها
- 1956_p080_l0012.png: نالتها -> نال بها
- 1964b_p077_l0044.png: السعودينة -> السعود نية
- 1964b_p077_l0044.png: بوجذفي -> توجد في
- 1955_p174_l0031.png: تردعليها -> ترد عليها
- 1964_p080_l0031.png: ورداتهم -> ورد اتهم
- 1958_p143_l0035.png: الفومي -> الف ومن
- 1954_p101_l0040.png: نمرابو -> نمر ابو
- 1955_p046_l0014.png: وخيرهم -> وخير هم
- 1964_p189_l0029.png: حولتها -> حول بها
- 1956_p089_l0024.png: الجزاز -> الخ زار
- 1960_p036_l0026.png: فالدول -> قال دول
- 1954_p043_l0013.png: الفعترة -> الف عبرة
- 1964_p046_l0068.png: سفرانها -> سفر انها
- 1964_p046_l0068.png: بالسكان -> بال سكان
- 1955_p126_l0019.png: اشترذكي -> استرد 

In [36]:
import pandas as pd, re, math
from collections import Counter
from tqdm import tqdm

# =========================
# PATHS (عدليها لو تبين)
# =========================
TRAIN_CSV = "/kaggle/input/datasets/geleefinal/gelee-nakba/AR-MS.train/annotations.csv"

A_PATH = "/kaggle/working/submission_split_attack.csv"              # ناتج split attack
B_PATH = "/kaggle/working/submission_lastshot.csv"                  # أفضل سابقًا 0.1138
C_PATH = "/kaggle/working/submission_last_attack_ultra.csv"         # ثاني أفضل

OUT_PATH = "/kaggle/working/submission_hybrid3_pick.csv"

# =========================
# Helpers
# =========================
WORD_RE = re.compile(r"[اأإآء-ي0-9٠-٩]+")
AR_NUMS = set("0123456789٠١٢٣٤٥٦٧٨٩")

def words(s):
    return [m.group(0) for m in WORD_RE.finditer(s)]

def basic_clean(s):
    s = re.sub(r"\s+", " ", str(s)).strip()
    s = re.sub(r"\s+([،\.!\?؛:])", r"\1", s)
    s = re.sub(r"([،\.!\?؛:])(?=\S)", r"\1 ", s)  # مسافة بعد الترقيم
    s = re.sub(r"\s+", " ", s).strip()
    return s

def rep_penalty(s):
    # عقوبة تكرار نفس الحرف كثير (يقتل WER/CER غالبًا)
    return len(re.findall(r"(.)\1{3,}", s))

def weird_char_penalty(s):
    # عقوبة كثرة الرموز الغريبة
    return sum(1 for ch in s if (not ch.isalnum()) and ch not in " ،.؟!؛:()[]«»\"'ـ-")

def digit_penalty(s):
    # عقوبة كلمات أرقام طويلة/غريبة (أحيانًا تزود الأخطاء)
    ws = words(s)
    p = 0
    for w in ws:
        if all(ch in AR_NUMS for ch in w):
            if len(w) >= 5: p += 1
    return p

# =========================
# Build Train Lexicon + bigram-ish counts
# =========================
train = pd.read_csv(TRAIN_CSV)
train_texts = train["text"].dropna().astype(str).tolist()

train_words = []
train_bigrams = Counter()

for t in train_texts:
    w = words(t)
    train_words.extend(w)
    for i in range(len(w)-1):
        train_bigrams[(w[i], w[i+1])] += 1

freq = Counter(train_words)
vocab = set(freq.keys())

print("✅ train unique words:", len(vocab))
print("✅ train total words :", sum(freq.values()))
print("✅ train bigrams     :", len(train_bigrams))

def score_line(s):
    s = basic_clean(s)
    ws = words(s)

    if len(ws) == 0:
        return -1e9, s

    oov = 0
    in_vocab = 0
    tf_score = 0.0

    for w in ws:
        if w in vocab:
            in_vocab += 1
            tf_score += math.log1p(freq[w])   # مكافأة تكرار الكلمة في التدريب
        else:
            oov += 1

    # مكافأة بسيطة للـ bigrams الشائعة
    bi = 0.0
    for i in range(len(ws)-1):
        c = train_bigrams.get((ws[i], ws[i+1]), 0)
        if c:
            bi += math.log1p(c)

    # عقوبات
    p_rep   = rep_penalty(s)
    p_weird = weird_char_penalty(s)
    p_dig   = digit_penalty(s)

    # طول منطقي: لا نبي سطر 1 كلمة إذا عادة السطور أطول
    short_pen = 0.0
    if len(ws) <= 1:
        short_pen -= 3.5
    elif len(ws) == 2:
        short_pen -= 1.0

    # OOV عقوبة قوية لأنها غالبًا أخطاء (لكن مو دايم)
    # نستخدم نسبة OOV بدل العدد فقط
    oov_ratio = oov / max(1, len(ws))

    score = 0.0
    score += 1.4 * tf_score
    score += 0.8 * bi
    score += -8.0 * oov_ratio
    score += short_pen
    score += -1.0 * p_rep
    score += -0.5 * p_weird
    score += -0.6 * p_dig

    # مكافأة خفيفة لعدد كلمات منطقي (بدون إفراط)
    score += 0.25 * min(len(ws), 12)

    return score, s

# =========================
# Load submissions
# =========================
A = pd.read_csv(A_PATH)
B = pd.read_csv(B_PATH)
C = pd.read_csv(C_PATH)

# توحيد حسب filename (مهم)
A = A[["filename","text"]].copy()
B = B[["filename","text"]].copy()
C = C[["filename","text"]].copy()

M = A.merge(B, on="filename", how="inner", suffixes=("_A","_B")).merge(
    C, on="filename", how="inner"
).rename(columns={"text":"text_C"})

print("✅ merged rows:", len(M))

# =========================
# Pick best per line
# =========================
picked = []
changed = 0
examples = []

for i, row in enumerate(tqdm(M.itertuples(index=False), total=len(M), desc="Hybrid3 pick")):
    fn = row.filename
    ta, tb, tc = row.text_A, row.text_B, row.text_C

    sa, ta2 = score_line(ta)
    sb, tb2 = score_line(tb)
    sc, tc2 = score_line(tc)

    # اختيار أقوى سكور
    # شرط أمان: لازم يتفوق بفارق واضح لو بيغير عن A
    best_s, best_t, best_src = sa, ta2, "A"

    if sb > best_s:
        best_s, best_t, best_src = sb, tb2, "B"
    if sc > best_s:
        best_s, best_t, best_src = sc, tc2, "C"

    # safety margin: لو الفائز مو A لازم يكون الفرق واضح عن A
    if best_src != "A":
        margin = best_s - sa
        if margin < 1.25:   # لو الفرق بسيط… خليه A
            best_s, best_t, best_src = sa, ta2, "A"
        else:
            changed += 1
            if len(examples) < 30:
                examples.append((fn, best_src, round(sa,3), round(sb,3), round(sc,3), ta2[:160], best_t[:160]))

    picked.append((fn, best_t))

out = pd.DataFrame(picked, columns=["filename","text"])
out.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print("\n✅ DONE Hybrid3")
print("Changed lines vs A:", changed, "/", len(out), f"({100*changed/len(out):.2f}%)")
print("Saved:", OUT_PATH)

print("\n🔎 Sample changed lines (first 30):")
for fn, src, sa, sb, sc, old, new in examples:
    print("-"*80)
    print("file:", fn, "| picked:", src, "| scores A/B/C:", sa, sb, sc)
    print("A:", old)
    print("NEW:", new)

✅ train unique words: 42270
✅ train total words : 159749
✅ train bigrams     : 118464
✅ merged rows: 2671


Hybrid3 pick: 100%|██████████| 2671/2671 [00:00<00:00, 7549.87it/s]


✅ DONE Hybrid3
Changed lines vs A: 607 / 2671 (22.73%)
Saved: /kaggle/working/submission_hybrid3_pick.csv

🔎 Sample changed lines (first 30):
--------------------------------------------------------------------------------
file: 1962_p077_l0032.png | picked: C | scores A/B/C: -2.447 -2.447 0.732
A: وي: المرع فالذركي تموبر عديرابر مشعل جيالاشبين شقبه ويالو وديرياسين
NEW: د احي سشوايدو المرعة الذبي شمو بر عدير برمشعل ا جما لانشبتين شقبه ويالو وديرياسين
--------------------------------------------------------------------------------
file: 1954_p068_l0020.png | picked: C | scores A/B/C: 75.545 75.545 79.447
A: ان امام هذه العاصفه احتمالات ١ اما ان تهدا بنفسها ٢ او انها تستخر في كفاحها حتي تنجح
NEW: إن أمام هته العاصفة أختالاتا ١(١) أما آن تهدا بنفسها (٢) أو أنها تستنرة لفاحها حتى تنجح
--------------------------------------------------------------------------------
file: 1955_p174_l0031.png | picked: C | scores A/B/C: 51.079 41.252 54.971
A: أتاني تلفونات فكانت الخادم ترد عليها بأني في عما

In [37]:
import pandas as pd, re, math
from collections import Counter, defaultdict
from tqdm import tqdm

# ===== paths =====
TRAIN_CSV = "/kaggle/input/datasets/geleefinal/gelee-nakba/AR-MS.train/annotations.csv"
IN_SUB    = "/kaggle/working/submission_lastshot.csv"
OUT_SUB   = "/kaggle/working/submission_microdot_aggressive.csv"

# ===== regex =====
WORD_RE = re.compile(r"[اأإآء-ي0-9٠-٩]+")

def words_with_spans(s):
    return [(m.group(0), m.start(), m.end()) for m in WORD_RE.finditer(str(s))]

# ===== undot map =====
DOT_BASE = {
    "ب":"ب","ت":"ب","ث":"ب",
    "ج":"ح","ح":"ح","خ":"ح",
    "د":"د","ذ":"د",
    "ر":"ر","ز":"ر",
    "س":"س","ش":"س",
    "ص":"ص","ض":"ص",
    "ط":"ط","ظ":"ط",
    "ع":"ع","غ":"ع",
    "ف":"ف","ق":"ف",
    "ن":"ن","ي":"ن","ى":"ن","ئ":"ن",
    "ه":"ه","ة":"ه",
}
def undot(w): return "".join(DOT_BASE.get(ch, ch) for ch in w)

# ===== build train vocab =====
train = pd.read_csv(TRAIN_CSV)
train_words = []
for t in train["text"].dropna().astype(str):
    train_words.extend([w for w,_,_ in words_with_spans(t)])

freq = Counter(train_words)
vocab = set(freq.keys())

# undot index: لكل رسم بدون نقاط، قائمة مرشحين مرتبين بالتكرار
undot_index = defaultdict(list)
for w,c in freq.items():
    undot_index[undot(w)].append((w,c))

for u in undot_index:
    undot_index[u].sort(key=lambda x: x[1], reverse=True)

print("✅ train unique words:", len(vocab))
print("✅ undot keys:", len(undot_index))

# ===== choose replacement (aggressive but bounded) =====
MIN_LEN = 4
MAX_DIFF = 2
MIN_FREQ = 3
FREQ_MARGIN = 1.2   # لازم البديل يكون أعلى تكرارًا بهذا العامل

def choose_replacement(w):
    if len(w) < MIN_LEN:
        return None
    if w in vocab:
        return None

    u = undot(w)
    cands = undot_index.get(u)
    if not cands:
        return None

    best = None
    best_gain = 0

    for cand, c_freq in cands[:12]:
        if len(cand) != len(w):
            continue

        # احسب عدد الاختلافات
        diffs = sum(1 for a,b in zip(w, cand) if a != b)
        if diffs == 0 or diffs > MAX_DIFF:
            continue

        if c_freq < MIN_FREQ:
            continue

        # شرط تفضيل قوي
        base_freq = freq.get(w, 0)
        if c_freq < (base_freq * FREQ_MARGIN + 1):
            continue

        gain = c_freq - base_freq
        if gain > best_gain:
            best_gain = gain
            best = cand

    return best

# ===== apply =====
sub = pd.read_csv(IN_SUB)

total_tokens = 0
changed = 0
examples = []

new_texts = []

for i, text in enumerate(tqdm(sub["text"].astype(str), desc="MicroDot Aggressive")):
    spans = words_with_spans(text)
    total_tokens += len(spans)

    chars = list(text)

    for w, s, e in reversed(spans):
        rep = choose_replacement(w)
        if rep and rep != w:
            chars[s:e] = list(rep)
            changed += 1
            if len(examples) < 25:
                examples.append((sub.loc[i, "filename"], w, rep))

    new_texts.append("".join(chars))

sub["text"] = new_texts
sub.to_csv(OUT_SUB, index=False, encoding="utf-8-sig")

print("\n✅ DONE MicroDot Aggressive")
print("Total tokens:", total_tokens)
print("Changed tokens:", changed)
print("Change rate:", round(100*changed/max(total_tokens,1),3), "%")
print("Saved:", OUT_SUB)

print("\n🔎 Sample changes:")
for fn, old, new in examples:
    print(f"- {fn}: {old} -> {new}")

✅ train unique words: 42270
✅ undot keys: 39334


MicroDot Aggressive: 100%|██████████| 2671/2671 [00:00<00:00, 51185.10it/s]


✅ DONE MicroDot Aggressive
Total tokens: 29587
Changed tokens: 602
Change rate: 2.035 %
Saved: /kaggle/working/submission_microdot_aggressive.csv

🔎 Sample changes:
- 1960_p116_l0063.png: ناخيه -> ناحية
- 1962_p077_l0032.png: شقبه -> شقبة
- 1958_p012_l0029.png: دنيار -> دينار
- 1962_p173_l0055.png: محكمه -> محكمة
- 1954_p068_l0020.png: تهدا -> بهذا
- 1954_p068_l0020.png: العاصفه -> العاصفة
- 1954_p178_l0021.png: الأفراج -> الأفراح
- 1958_p143_l0035.png: الفومي -> القومي
- 1964_p193_l0050.png: شاريا -> ساريا
- 1964_p189_l0029.png: عربيه -> عربية
- 1956_p089_l0024.png: بكلمه -> بكلمة
- 1964_p046_l0068.png: بقطع -> تقطع
- 1962_p080_l0024.png: خطوه -> خطوة
- 1954_p059_l0035.png: بسيطه -> بسيطة
- 1964b_p098_l0044.png: الأثات -> الأثاث
- 1958_p143_l0036.png: الساي -> الشاي
- 1959_p103_l0051.png: حوالى -> حوالي
- 1962_p154_l0053.png: يحملوي -> يحملون
- 1954_p102_l0014.png: وجهى -> وجهي
- 1964_p159_l0040.png: برقي -> ترقى
- 1964_p010_l0060.png: دقيقه -> دقيقة
- 1962_p195_l0062.png: الصعفاء ->

In [38]:
import pandas as pd, re
from collections import Counter, defaultdict
from tqdm import tqdm

# ===== paths =====
TRAIN_CSV = "/kaggle/input/datasets/geleefinal/gelee-nakba/AR-MS.train/annotations.csv"
IN_SUB    = "/kaggle/working/submission_lastshot.csv"
OUT_SUB   = "/kaggle/working/submission_microdot_ultra_v2.csv"

WORD_RE = re.compile(r"[اأإآء-ي0-9٠-٩]+")

def words_with_spans(s):
    return [(m.group(0), m.start(), m.end()) for m in WORD_RE.finditer(str(s))]

# ===== undot map =====
DOT_BASE = {
    "ب":"ب","ت":"ب","ث":"ب",
    "ج":"ح","ح":"ح","خ":"ح",
    "د":"د","ذ":"د",
    "ر":"ر","ز":"ر",
    "س":"س","ش":"س",
    "ص":"ص","ض":"ص",
    "ط":"ط","ظ":"ط",
    "ع":"ع","غ":"ع",
    "ف":"ف","ق":"ف",
    "ن":"ن","ي":"ن","ى":"ن","ئ":"ن",
    "ه":"ه","ة":"ه",
}
def undot(w): return "".join(DOT_BASE.get(ch, ch) for ch in w)

# ===== build train vocab =====
train = pd.read_csv(TRAIN_CSV)
train_words = []
for t in train["text"].dropna().astype(str):
    train_words.extend([w for w,_,_ in words_with_spans(t)])

freq = Counter(train_words)
vocab = set(freq.keys())

undot_index = defaultdict(list)
for w,c in freq.items():
    undot_index[undot(w)].append((w,c))
for u in undot_index:
    undot_index[u].sort(key=lambda x: x[1], reverse=True)

print("✅ train unique words:", len(vocab))
print("✅ undot keys:", len(undot_index))

# ===== ULTRA tuning knobs =====
MIN_LEN = 4
MAX_DIFF = 2
MIN_FREQ = 2

FREQ_MARGIN = 1.02   # 👈 الضغط الحقيقي
ALLOW_REPLACE_IN_VOCAB_IF_WEAK = True
WEAK_RATIO = 0.60    # إذا الكلمة موجودة بالـ vocab لكن تكرارها أقل من 60% من المرشح الأقوى نسمح بالتبديل
TOPK = 25            # عدد المرشحين اللي نفحصهم لكل undot key

def choose_replacement(w):
    if len(w) < MIN_LEN:
        return None

    u = undot(w)
    cands = undot_index.get(u)
    if not cands:
        return None

    w_freq = freq.get(w, 0)

    # إذا الكلمة موجودة في التدريب: عادة لا نلمسها
    # لكن (Ultra) نلمسها إذا كانت "ضعيفة" مقارنة بأفضل مرشح
    if w in vocab and not ALLOW_REPLACE_IN_VOCAB_IF_WEAK:
        return None

    best = None
    best_score = -1e18

    # أقوى مرشح (لشرط WEAK_RATIO)
    top_word, top_freq = cands[0]

    if w in vocab and ALLOW_REPLACE_IN_VOCAB_IF_WEAK:
        # لو w_freq قريب من top_freq ما نلمسها
        if top_freq <= 0 or (w_freq / top_freq) >= WEAK_RATIO:
            return None

    for cand, c_freq in cands[:TOPK]:
        if len(cand) != len(w):
            continue
        if c_freq < MIN_FREQ:
            continue

        diffs = sum(1 for a,b in zip(w, cand) if a != b)
        if diffs == 0 or diffs > MAX_DIFF:
            continue

        # شرط تفضيل: المرشح يكون أعلى من الحالي
        # لو w_freq=0 (OOV) نكتفي بـ MIN_FREQ
        if w_freq > 0:
            if c_freq < (w_freq * FREQ_MARGIN + 1):
                continue

        # score: نفضّل التكرار العالي + اختلافات أقل
        score = (c_freq * 1.0) - (diffs * 0.75)
        if score > best_score:
            best_score = score
            best = cand

    return best

# ===== apply =====
sub = pd.read_csv(IN_SUB)

total_tokens = 0
changed = 0
examples = []
new_texts = []

for i, text in enumerate(tqdm(sub["text"].astype(str), desc="MicroDot ULTRA v2")):
    spans = words_with_spans(text)
    total_tokens += len(spans)

    chars = list(text)

    for w, s, e in reversed(spans):
        rep = choose_replacement(w)
        if rep and rep != w:
            chars[s:e] = list(rep)
            changed += 1
            if len(examples) < 35:
                examples.append((sub.loc[i, "filename"], w, rep))

    new_texts.append("".join(chars))

sub["text"] = new_texts
sub.to_csv(OUT_SUB, index=False, encoding="utf-8-sig")

print("\n✅ DONE MicroDot ULTRA v2")
print("Total tokens:", total_tokens)
print("Changed tokens:", changed)
print("Change rate:", round(100*changed/max(total_tokens,1),3), "%")
print("Saved:", OUT_SUB)

print("\n🔎 Sample changes:")
for fn, old, new in examples:
    print(f"- {fn}: {old} -> {new}")

✅ train unique words: 42270
✅ undot keys: 39334


MicroDot ULTRA v2: 100%|██████████| 2671/2671 [00:00<00:00, 33776.90it/s]


✅ DONE MicroDot ULTRA v2
Total tokens: 29587
Changed tokens: 1019
Change rate: 3.444 %
Saved: /kaggle/working/submission_microdot_ultra_v2.csv

🔎 Sample changes:
- 1960_p116_l0063.png: ناخيه -> ناحية
- 1960_p116_l0063.png: يغته -> نعته
- 1962_p077_l0032.png: شقبه -> شقبة
- 1958_p012_l0029.png: دنيار -> دينار
- 1962_p173_l0055.png: واقر -> وافر
- 1962_p173_l0055.png: محكمه -> محكمة
- 1954_p068_l0020.png: تهدا -> بهذا
- 1954_p068_l0020.png: العاصفه -> العاصفة
- 1954_p178_l0021.png: الأفراج -> الأفراح
- 1958_p143_l0035.png: الفومي -> القومي
- 1953b_p001_l0015.png: السهر -> الشهر
- 1964_p193_l0050.png: شاريا -> ساريا
- 1964_p193_l0050.png: لقمه -> لقمة
- 1960_p033_l0064.png: السر -> الشر
- 1955_p062_l0038.png: الحزب -> الحرب
- 1964_p189_l0029.png: عربيه -> عربية
- 1956_p089_l0024.png: بكلمه -> بكلمة
- 1964_p046_l0068.png: بقطع -> تقطع
- 1962_p080_l0024.png: خطوه -> خطوة
- 1962_p080_l0024.png: بقدر -> تقدر
- 1954_p059_l0035.png: بسيطه -> بسيطة
- 1954_p059_l0035.png: سهره -> سهرة
- 1960_p10

In [39]:
import pandas as pd, re
from collections import Counter, defaultdict
from tqdm import tqdm

TRAIN_CSV = "/kaggle/input/datasets/geleefinal/gelee-nakba/AR-MS.train/annotations.csv"
IN_SUB    = "/kaggle/working/submission_lastshot.csv"
OUT_SUB   = "/kaggle/working/submission_microdot_mid15.csv"

WORD_RE = re.compile(r"[اأإآء-ي0-9٠-٩]+")

def words_with_spans(s):
    return [(m.group(0), m.start(), m.end()) for m in WORD_RE.finditer(str(s))]

DOT_BASE = {
    "ب":"ب","ت":"ب","ث":"ب",
    "ج":"ح","ح":"ح","خ":"ح",
    "د":"د","ذ":"د",
    "ر":"ر","ز":"ر",
    "س":"س","ش":"س",
    "ص":"ص","ض":"ص",
    "ط":"ط","ظ":"ط",
    "ع":"ع","غ":"ع",
    "ف":"ف","ق":"ف",
    "ن":"ن","ي":"ن","ى":"ن","ئ":"ن",
    "ه":"ه","ة":"ه",
}
def undot(w): return "".join(DOT_BASE.get(ch, ch) for ch in w)

# ---- train vocab ----
train = pd.read_csv(TRAIN_CSV)
train_words = []
for t in train["text"].dropna().astype(str):
    train_words.extend([w for w,_,_ in words_with_spans(t)])

freq = Counter(train_words)
vocab = set(freq.keys())

undot_index = defaultdict(list)
for w,c in freq.items():
    undot_index[undot(w)].append((w,c))
for u in undot_index:
    undot_index[u].sort(key=lambda x: x[1], reverse=True)

print("✅ train unique words:", len(vocab))
print("✅ undot keys:", len(undot_index))

# ---- MID v1.5 knobs ----
MIN_LEN = 4
MAX_DIFF = 2
MIN_FREQ = 3

FREQ_MARGIN = 1.10
TOPK = 15
ALLOW_REPLACE_IN_VOCAB = False   # 👈 أهم فرق عن ultra

def choose_replacement(w):
    if len(w) < MIN_LEN:
        return None
    if (w in vocab) and (not ALLOW_REPLACE_IN_VOCAB):
        return None

    u = undot(w)
    cands = undot_index.get(u)
    if not cands:
        return None

    w_freq = freq.get(w, 0)

    best = None
    best_score = -1e18

    for cand, c_freq in cands[:TOPK]:
        if len(cand) != len(w):
            continue
        if c_freq < MIN_FREQ:
            continue

        diffs = sum(1 for a,b in zip(w, cand) if a != b)
        if diffs == 0 or diffs > MAX_DIFF:
            continue

        if w_freq > 0:
            if c_freq < (w_freq * FREQ_MARGIN + 1):
                continue

        score = (c_freq * 1.0) - (diffs * 0.6)  # خفّفنا العقوبة شوي عن ultra
        if score > best_score:
            best_score = score
            best = cand

    return best

# ---- apply ----
sub = pd.read_csv(IN_SUB)

total_tokens = 0
changed = 0
examples = []
new_texts = []

for i, text in enumerate(tqdm(sub["text"].astype(str), desc="MicroDot MID v1.5")):
    spans = words_with_spans(text)
    total_tokens += len(spans)

    chars = list(text)
    for w, s, e in reversed(spans):
        rep = choose_replacement(w)
        if rep and rep != w:
            chars[s:e] = list(rep)
            changed += 1
            if len(examples) < 30:
                examples.append((sub.loc[i, "filename"], w, rep))

    new_texts.append("".join(chars))

sub["text"] = new_texts
sub.to_csv(OUT_SUB, index=False, encoding="utf-8-sig")

print("\n✅ DONE MicroDot MID v1.5")
print("Total tokens:", total_tokens)
print("Changed tokens:", changed)
print("Change rate:", round(100*changed/max(total_tokens,1),3), "%")
print("Saved:", OUT_SUB)

print("\n🔎 Sample changes:")
for fn, old, new in examples:
    print(f"- {fn}: {old} -> {new}")

✅ train unique words: 42270
✅ undot keys: 39334


MicroDot MID v1.5: 100%|██████████| 2671/2671 [00:00<00:00, 49874.84it/s]


✅ DONE MicroDot MID v1.5
Total tokens: 29587
Changed tokens: 602
Change rate: 2.035 %
Saved: /kaggle/working/submission_microdot_mid15.csv

🔎 Sample changes:
- 1960_p116_l0063.png: ناخيه -> ناحية
- 1962_p077_l0032.png: شقبه -> شقبة
- 1958_p012_l0029.png: دنيار -> دينار
- 1962_p173_l0055.png: محكمه -> محكمة
- 1954_p068_l0020.png: تهدا -> بهذا
- 1954_p068_l0020.png: العاصفه -> العاصفة
- 1954_p178_l0021.png: الأفراج -> الأفراح
- 1958_p143_l0035.png: الفومي -> القومي
- 1964_p193_l0050.png: شاريا -> ساريا
- 1964_p189_l0029.png: عربيه -> عربية
- 1956_p089_l0024.png: بكلمه -> بكلمة
- 1964_p046_l0068.png: بقطع -> تقطع
- 1962_p080_l0024.png: خطوه -> خطوة
- 1954_p059_l0035.png: بسيطه -> بسيطة
- 1964b_p098_l0044.png: الأثات -> الأثاث
- 1958_p143_l0036.png: الساي -> الشاي
- 1959_p103_l0051.png: حوالى -> حوالي
- 1962_p154_l0053.png: يحملوي -> يحملون
- 1954_p102_l0014.png: وجهى -> وجهي
- 1964_p159_l0040.png: برقي -> ترقى
- 1964_p010_l0060.png: دقيقه -> دقيقة
- 1962_p195_l0062.png: الصعفاء -> الضعفا

In [40]:
import pandas as pd, re
from collections import Counter, defaultdict
from tqdm import tqdm

TRAIN_CSV = "/kaggle/input/datasets/geleefinal/gelee-nakba/AR-MS.train/annotations.csv"
IN_SUB    = "/kaggle/working/submission_lastshot.csv"
OUT_SUB   = "/kaggle/working/submission_lastshot_sniper.csv"

WORD_RE = re.compile(r"[اأإآء-ي0-9٠-٩]+")

def words_with_spans(s):
    return [(m.group(0), m.start(), m.end()) for m in WORD_RE.finditer(str(s))]

DOT_BASE = {
    "ب":"ب","ت":"ب","ث":"ب",
    "ج":"ح","ح":"ح","خ":"ح",
    "د":"د","ذ":"د",
    "ر":"ر","ز":"ر",
    "س":"س","ش":"س",
    "ص":"ص","ض":"ص",
    "ط":"ط","ظ":"ط",
    "ع":"ع","غ":"ع",
    "ف":"ف","ق":"ف",
    "ن":"ن","ي":"ن","ى":"ن","ئ":"ن",
    "ه":"ه","ة":"ه",
}
def undot(w): return "".join(DOT_BASE.get(ch, ch) for ch in w)

# ---- train vocab ----
train = pd.read_csv(TRAIN_CSV)
train_words = []
for t in train["text"].dropna().astype(str):
    train_words.extend([w for w,_,_ in words_with_spans(t)])

freq = Counter(train_words)
vocab = set(freq.keys())

undot_index = defaultdict(list)
for w,c in freq.items():
    undot_index[undot(w)].append((w,c))
for u in undot_index:
    undot_index[u].sort(key=lambda x: x[1], reverse=True)

print("✅ train unique words:", len(vocab))
print("✅ undot keys:", len(undot_index))

# ---- SNIPER knobs ----
MIN_LEN = 7          # 👈 انتقائي: كلمات طويلة فقط
MAX_DIFF = 2
MIN_FREQ = 4
FREQ_MARGIN = 1.25   # لازم فرق قوي بالتكرار
TOPK = 20

def choose_replacement(w):
    if len(w) < MIN_LEN:
        return None
    if w in vocab:
        return None  # لا نلمس كلمة تدريبية

    u = undot(w)
    cands = undot_index.get(u)
    if not cands:
        return None

    w_freq = freq.get(w, 0)

    best, best_score = None, -1e18
    for cand, c_freq in cands[:TOPK]:
        if len(cand) != len(w): 
            continue
        if c_freq < MIN_FREQ:
            continue

        diffs = sum(1 for a,b in zip(w, cand) if a != b)
        if diffs == 0 or diffs > MAX_DIFF:
            continue

        # لازم المرشح يكون أقوى بوضوح
        if w_freq > 0 and c_freq < (w_freq * FREQ_MARGIN + 1):
            continue

        # score: تكرار عالي + اختلافات قليلة
        score = (c_freq * 1.0) - (diffs * 0.8)
        if score > best_score:
            best_score = score
            best = cand

    return best

# ---- apply ----
sub = pd.read_csv(IN_SUB)

total_tokens = 0
changed = 0
examples = []
new_texts = []

for i, text in enumerate(tqdm(sub["text"].astype(str), desc="SNIPER dot-fix")):
    spans = words_with_spans(text)
    total_tokens += len(spans)

    chars = list(text)
    for w, s, e in reversed(spans):
        rep = choose_replacement(w)
        if rep and rep != w:
            chars[s:e] = list(rep)
            changed += 1
            if len(examples) < 35:
                examples.append((sub.loc[i, "filename"], w, rep))

    new_texts.append("".join(chars))

sub["text"] = new_texts
sub.to_csv(OUT_SUB, index=False, encoding="utf-8-sig")

print("\n✅ DONE SNIPER dot-fix")
print("Total tokens:", total_tokens)
print("Changed tokens:", changed)
print("Change rate:", round(100*changed/max(total_tokens,1),3), "%")
print("Saved:", OUT_SUB)

print("\n🔎 Sample changes:")
for fn, old, new in examples:
    print(f"- {fn}: {old} -> {new}")

✅ train unique words: 42270
✅ undot keys: 39334


SNIPER dot-fix: 100%|██████████| 2671/2671 [00:00<00:00, 65727.87it/s]


✅ DONE SNIPER dot-fix
Total tokens: 29587
Changed tokens: 122
Change rate: 0.412 %
Saved: /kaggle/working/submission_lastshot_sniper.csv

🔎 Sample changes:
- 1954_p068_l0020.png: العاصفه -> العاصفة
- 1954_p178_l0021.png: الأفراج -> الأفراح
- 1962_p195_l0062.png: الصعفاء -> الضعفاء
- 1964_p188_l0034.png: المقدسه -> المقدسة
- 1956_p051_l0046.png: العداله -> العدالة
- 1962_p091_l0010.png: الاشتراكيه -> الاشتراكية
- 1962_p116_l0051.png: لمقاومه -> لمقاومة
- 1954_p181_l0043.png: القراءه -> القراءة
- 1958_p021_l0038.png: العربيه -> العربية
- 1955_p062_l0005.png: العراقيه -> العراقية
- 1955_p072_l0014.png: الأفطار -> الأقطار
- 1964b_p098_l0025.png: الوكاله -> الوكالة
- 1954_p013_l0024.png: الدعايه -> الدعاية
- 1955_p068_l0011.png: الثانيه -> الثانية
- 1962_p086_l0034.png: الانكليزيه -> الانكليزية
- 1962_p134_l0015.png: الموسيقي -> الموسيقى
- 1964_p159_l0027.png: والبيره -> والبيرة
- 1964_p159_l0027.png: البراغثه -> البراغثة
- 1955_p028_l0021.png: الأحراب -> الأحزاب
- 1962_p192_l0043.png: الث

In [41]:
import pandas as pd, re, os
from collections import Counter
from tqdm import tqdm

# ===== paths (عدلي/زيدي براحتك) =====
TRAIN_CSV = "/kaggle/input/datasets/geleefinal/gelee-nakba/AR-MS.train/annotations.csv"

CANDIDATES = [
    "/kaggle/working/submission_lastshot.csv",             # أفضل CER عندك 0.1133
    "/kaggle/working/submission_last_attack_ultra.csv",    # ثاني أفضل
    "/kaggle/input/datasets/geleefinal/top-reading/submission (8).csv",  # قراءة قوية (قديمة)
    "/kaggle/input/datasets/geleefinal/best-2/Final_Submission_Rank1.csv",
    "/kaggle/input/datasets/geleefinal/best-3/Final_Submission_Gold_V5.csv",
    "/kaggle/input/datasets/geleefinal/best-4/LAST.csv",
    "/kaggle/input/datasets/geleefinal/best-5/submission (10).csv",
]
OUT_PATH = "/kaggle/working/submission_ensemble_END.csv"

# ===== helpers =====
WORD_RE = re.compile(r"[اأإآء-ي0-9٠-٩]+")
def words(s): return WORD_RE.findall(str(s))

def levenshtein(a, b):
    a = str(a); b = str(b)
    if a == b: return 0
    if len(a) < len(b):
        a, b = b, a
    # now len(a) >= len(b)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            ins = cur[j-1] + 1
            dele = prev[j] + 1
            sub = prev[j-1] + (ca != cb)
            cur.append(min(ins, dele, sub))
        prev = cur
    return prev[-1]

# ===== build train vocab =====
train_df = pd.read_csv(TRAIN_CSV)
train_words = []
for t in train_df["text"].dropna().astype(str):
    train_words.extend(words(t))
train_vocab = set(train_words)
train_freq  = Counter(train_words)

print("✅ train unique words:", len(train_vocab))

# ===== read candidates & align by filename =====
dfs = []
for p in CANDIDATES:
    if not os.path.exists(p):
        print("⚠️ missing:", p)
        continue
    d = pd.read_csv(p)
    # handle if someone has header "filename,text" already
    if "filename" not in d.columns or "text" not in d.columns:
        # try common alternative
        cols = d.columns.tolist()
        raise ValueError(f"Bad columns in {p}: {cols}")
    dfs.append((p, d.set_index("filename")["text"].astype(str)))

assert len(dfs) >= 2, "لازم على الأقل ملفين للـ ensemble"

# filenames intersection (آمن)
common = set(dfs[0][1].index)
for _, s in dfs[1:]:
    common &= set(s.index)
common = sorted(list(common))
print("✅ common files:", len(common), "/", len(dfs[0][1].index))

# ===== scoring =====
def score_candidate(text, others):
    # 1) agreement score: negative mean edit distance (normalize قليلاً)
    if not others:
        agree = 0.0
    else:
        dists = [levenshtein(text, o) / max(1, len(o)) for o in others]
        agree = - (sum(dists) / len(dists))

    ws = words(text)
    if len(ws) == 0:
        in_ratio = 0.0
        oov_cnt = 0
    else:
        in_cnt = sum(1 for w in ws if w in train_vocab)
        oov_cnt = len(ws) - in_cnt
        in_ratio = in_cnt / len(ws)

    # 2) language prior: prefer higher in-vocab ratio + prefer frequent words (خفيف)
    freq_bonus = 0.0
    for w in ws[:30]:  # ما نطوّل
        freq_bonus += min(3.0, (train_freq.get(w, 0) ** 0.5) / 20.0)

    # 3) mild penalties
    penalty = 0.0
    if len(text) < 3: penalty -= 1.5
    if len(text) > 220: penalty -= 0.5
    penalty -= 0.08 * oov_cnt  # OOV hurts WER غالباً

    # weights
    return (1.6 * agree) + (1.1 * in_ratio) + (0.15 * freq_bonus) + penalty

picked = []
debug_samples = 0

for i, fn in enumerate(tqdm(common, desc="Ensembling")):
    cands = []
    for p, s in dfs:
        cands.append((p, s.loc[fn]))

    # compute best
    best_p, best_t, best_sc = None, None, -1e18
    for idx, (p, t) in enumerate(cands):
        others = [tt for j,(pp,tt) in enumerate(cands) if j != idx]
        sc = score_candidate(t, others)
        if sc > best_sc:
            best_sc, best_p, best_t = sc, p, t

    picked.append((fn, best_t))

    # طباعة بسيطة أول كم مثال
    if debug_samples < 3:
        print("\n—"*80)
        print("File:", fn)
        for p, t in cands[:4]:
            print(" •", os.path.basename(p), ":", t[:120])
        print("✅ chosen:", os.path.basename(best_p), "| score:", round(best_sc, 4))
        debug_samples += 1

out = pd.DataFrame(picked, columns=["filename","text"])
out.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print("\n✅ Saved:", OUT_PATH)
print(out.head())

✅ train unique words: 42270
✅ common files: 2671 / 2671


Ensembling:   0%|          | 0/2671 [00:00<?, ?it/s]


—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
File: 1953a_p007_l0024.png
 • submission_lastshot.csv : بعثة أخرى من فوق راسي وتخطوا الشرف والتقاليد الدولية والوفاء وهؤلاء كم قد موالي
 • submission_last_attack_ultra.csv : بصثة أخرى منفوق رأسي وتخطوا الشرف والتقاليد الدولية والوفاء وهؤلاء كم قدموا لي
 • submission (8).csv : بصثة أخرى منفوق رأسي وتخطوا الشرف والتقاليد الدولية والوفاء وهؤلاء كم قدموا لي
 • Final_Submission_Rank1.csv : بعثه اخري من فوق راسي وتخطوا الشرف والتقاليد الدوليه والوفاء وهؤلاء كم قدموا لي
✅ chosen: submission_lastshot.csv | score: 1.4582


Ensembling:   0%|          | 4/2671 [00:00<02:36, 17.01it/s]


—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
File: 1953a_p018_l0010.png
 • submission_lastshot.csv : ثم انتقلنا الي النادي وقد انصرف كثير من رواده لان شجارا نشب بين الدكتور
 • submission_last_attack_ultra.csv : ثم انتقلنا إلى النادي وقد انصرف كثير من رواده لأن شجارا لشعب بين الدكتور
 • submission (8).csv : ثم انتقلنا إلى النادي وقد انصرف كثير من رواده لأن شجارا لشعب بين الدكتور
 • Final_Submission_Rank1.csv : ثم انتقلنا الي النادي وقد انصرف كثير من رواده لان شجارا نشب بين الدكتور
✅ chosen: submission_last_attack_ultra.csv | score: 2.2298

—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
—
File: 1953a_p018_l0020.png
 • submission_lastshot.csv : أيدي افراد أو جماعات (شركات )بل يجب أن تسيطر عليها الحكومة حتى لا تستغل
 • submission_last_attack_ultra.csv : أيدي افراد أو

Ensembling: 100%|██████████| 2671/2671 [02:08<00:00, 20.84it/s]


✅ Saved: /kaggle/working/submission_ensemble_END.csv
               filename                                               text
0  1953a_p007_l0024.png  بعثة أخرى من فوق راسي وتخطوا الشرف والتقاليد ا...
1  1953a_p018_l0010.png  ثم انتقلنا إلى النادي وقد انصرف كثير من رواده ...
2  1953a_p018_l0020.png  أيدي افراد أو جماعات (شركات )بل يجب أن تسيطر ع...
3  1953a_p018_l0023.png  أما المعسكر الرأسمالي فإنه يشجع الربح والأفراد...
4  1953a_p018_l0026.png  في الخارج وإذا لم يصرفوه اضطروا لا يقاف دولاب ...
